# Data Science Case Studies: Structured Thinking for Top Product Company Interviews

---

## Why This Notebook Exists

In top product company interviews (Meta, Google, Netflix, Uber, Airbnb, Spotify), the **final answer is rarely what matters**. Interviewers evaluate:

1. **How you decompose ambiguity** — Can you turn a vague business problem into a tractable analytical framework?
2. **What questions you ask** — Do you identify hidden assumptions, edge cases, and stakeholder constraints?
3. **How you structure your approach** — Is your methodology principled, or are you pattern-matching to textbook solutions?
4. **How you communicate tradeoffs** — Can you articulate why one approach dominates another *in this specific context*?

---

## The Universal Framework (Before Diving Into Cases)

Every case study should be attacked with this mental model:

| Phase | What You Do | What It Signals |
| --- | --- | --- |
| **1. Clarify** | Ask 3-5 pointed questions about scope, constraints, stakeholders | You don't jump to solutions |
| **2. Frame** | Restate the problem in your own words with explicit assumptions | You understand the *actual* problem |
| **3. Decompose** | Break into sub-problems; identify what's measurable vs. what needs proxying | You think in systems |
| **4. Propose** | Lay out 2-3 candidate approaches with pros/cons | You have breadth |
| **5. Deep-dive** | Pick one approach and go deep on methodology | You have depth |
| **6. Validate** | Discuss how you'd know if your solution is working/failing | You think about production reality |

---

*Each case study below simulates a real interview scenario. The problem statement is intentionally vague — just like in an actual interview.*

# Case Study 1: Netflix — "Is Our Recommendation Engine Cannibalizing Content?"

---

## The Problem Statement (As Given by the Interviewer)

> *"Netflix just launched a major original series. The recommendation algorithm is heavily promoting it. Product leadership suspects that this is cannibalizing viewership of our long-tail catalog. How would you investigate this?"*

---

## Phase 1: Clarifying Questions (The First 3-5 Minutes)

An ideal candidate **does not** immediately start talking about A/B tests or causal inference. They ask:

### Questions That Reveal Depth of Thinking:

1. **"How are we defining cannibalization?"**
   - Is it total watch-hours shifting? Or is it specific genres being affected? Or is it that users who *would have* explored new content are now funneling into one show?
   - *Why this matters*: The definition changes the entire measurement strategy.

2. **"What's the time horizon of concern?"**
   - A show launched 2 days ago behaves very differently from one launched 2 months ago. Short-term spikes are expected — the concern is whether *long-term engagement patterns* are permanently altered.

3. **"Is the promotion algorithm-driven or editorially curated?"**
   - If it's algorithmic, we can study the counterfactual (users who were/weren't shown the recommendation). If editorial, we need a different identification strategy.

4. **"What does 'long-tail catalog' mean here? Which segments?"**
   - Is leadership worried about licensed content (revenue implications from contractual minimum views)? Independent films? International content?

5. **"Do we have a holdout group or is this a full rollout?"**
   - This determines whether we can use experimental data or must rely on observational methods.

---

## Phase 2: Problem Framing

### Restating the Problem

> "We want to understand whether heavy promotion of Show X has **caused** a reduction in engagement with other catalog content, or whether the observed decline (if any) is due to natural seasonality, content aging, or user preference evolution."

### Key Insight an Ideal Candidate States Explicitly:

> "This is fundamentally a **causal inference problem**, not a correlation problem. We can't just look at whether long-tail views dropped after launch — they might have dropped anyway. We need to estimate the **counterfactual**: what would viewership of the long-tail have been *in the absence of* the heavy promotion?"

---

## Phase 3: Decomposition Into Sub-Problems

### Sub-Problem A: Define the Outcome Metric

| Candidate Metric | Pros | Cons |
| --- | --- | --- |
| Total watch-hours on non-Show-X content | Simple, intuitive | Doesn't account for total time budget growth |
| Share of watch-hours on long-tail content | Accounts for growing pie | Share can drop even if absolute hours are stable |
| Content diversity index (e.g., Gini coefficient of user viewing) | Captures breadth of engagement | Hard to communicate to stakeholders |
| Unique titles viewed per user per week | Directly measures exploration | Doesn't weight depth of engagement |

**Ideal candidate's choice**: Use **multiple metrics** but lead with "unique titles viewed per user per week" as primary (directly measures exploration) and "share of watch-hours on non-promoted content" as secondary.

### Sub-Problem B: Establish the Counterfactual

Three approaches, ordered by strength:

1. **Experimental (Gold Standard)**: If a holdout exists where users were *not* shown heavy promotion of Show X, compare the two groups.

2. **Quasi-Experimental**: Use users who were "eligible" for the promotion but happened not to see it (e.g., they logged in via a device where the banner didn't render). Instrument variable or regression discontinuity design.

3. **Observational with Synthetic Control**: Build a synthetic counterfactual for each user's viewing behavior based on their pre-launch trajectory. Compare actual post-launch behavior to the synthetic prediction.

### Sub-Problem C: Distinguish Substitution from Complementarity

> "It's possible that Show X actually *grows* the pie — users who discover it might stay longer and then explore other content. The 'cannibalization' framing assumes a zero-sum time budget, which may not hold."

Measure:
- **Substitution**: Did users *replace* time on other content with Show X?
- **Complementarity**: Did users *add* Show X on top of their normal viewing?
- **Mixed**: Did new users come for Show X and then explore further?

---

## Phase 4: Proposed Approaches

### Approach 1: Difference-in-Differences (if holdout exists)

$$\tau = (\bar{Y}_{\text{treated, post}} - \bar{Y}_{\text{treated, pre}}) - (\bar{Y}_{\text{control, post}} - \bar{Y}_{\text{control, pre}})$$

Where $$Y$$ = unique titles viewed per user per week.

**Assumption to verify**: Parallel trends pre-launch. Plot both groups' metrics for 4-8 weeks before launch.

### Approach 2: Interrupted Time Series (no holdout)

Fit a model to pre-launch trend, project forward, compare to actual:

$$Y_t = \beta_0 + \beta_1 t + \beta_2 \cdot \mathbf{1}(t > t_{\text{launch}}) + \beta_3 \cdot (t - t_{\text{launch}}) \cdot \mathbf{1}(t > t_{\text{launch}}) + \epsilon_t$$

- $$\beta_2$$ captures the immediate level shift.
- $$\beta_3$$ captures the change in trend post-launch.

### Approach 3: User-Level Panel Regression

$$Y_{it} = \alpha_i + \gamma_t + \delta \cdot \text{Exposure}_{it} + X_{it}\beta + \epsilon_{it}$$

User fixed effects ($$\alpha_i$$) absorb time-invariant user preferences. Time fixed effects ($$\gamma_t$$) absorb seasonality. $$\text{Exposure}_{it}$$ is the degree to which user $$i$$ was exposed to Show X promotion at time $$t$$.

---

## Phase 5: Deep-Dive on Chosen Approach

*An ideal candidate picks one (say Approach 3) and goes deep:*

### Identification Concerns:
- **Selection bias**: Users who see the promotion more may be heavier users anyway. Need an instrument (e.g., algorithmic score threshold) or propensity score matching.
- **Simultaneity**: Users who are losing interest in long-tail *might be* the same ones the algorithm targets for the new show. The algorithm creates endogeneity.

### Practical Considerations:
- Use a 4-week pre-period and 4-week post-period.
- Exclude users who joined *after* launch (they have no counterfactual baseline).
- Cluster standard errors at the user level (repeated observations are correlated).

---

## Phase 6: Validation & Communication

### How Do We Know If We're Right?
- **Placebo tests**: Run the same analysis on a previous period where no major show launched. The estimated effect should be zero.
- **Dose-response**: If cannibalization is real, users who watched *more* of Show X should show *greater* decline in diversity. Check for monotone relationship.

### Communicating to Leadership:
> "We found that heavy promotion reduced per-user title diversity by X% in the first 4 weeks, but this effect dissipated by week 6. Users who binged the show actually increased exploration in weeks 5-8, suggesting a **temporary substitution** rather than permanent cannibalization. Recommendation: maintain promotion but introduce 'post-binge discovery' nudges."

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Mentions A/B testing | Acknowledges that the promotion may already be fully rolled out and proposes observational alternatives |
| Defines a metric | Proposes multiple metrics and explains why different stakeholders care about different ones |
| Runs a regression | Discusses endogeneity and why naive regression gives biased estimates |
| Reports the effect | Frames the business recommendation (temporary vs. permanent, action items) |

# Case Study 2: Uber — "Design a Better Surge Pricing Model"

---

## The Problem Statement (As Given by the Interviewer)

> *"Uber's current surge pricing multiplier is seen as a blunt instrument — riders hate it, and drivers game it. Leadership wants a 'smarter' pricing model that balances supply-demand while minimizing rider churn. How would you approach redesigning this?"*

---

## Phase 1: Clarifying Questions

### The Questions That Separate Senior From Junior:

1. **"What's the primary objective function? Minimize wait time? Maximize completed rides? Maximize revenue? Minimize rider churn?"**
   - *Why critical*: These objectives **conflict**. Higher prices reduce demand (shorter waits, fewer completed rides) but increase revenue per ride and driver supply. You cannot optimize all simultaneously.

2. **"What's the current surge mechanism? Is it zone-based, continuous, or tiered?"**
   - Understanding the status quo reveals what "better" means. If current system is zone-based, maybe the zones are too coarse. If continuous, maybe the volatility is the problem.

3. **"What data do we have on price elasticity? Do we know how rider behavior changes at 1.5x vs 2x vs 3x?"**
   - This tells us whether we need to *estimate* elasticity first or can assume we already know the demand curve.

4. **"What constraints exist? Regulatory caps on surge? PR-sensitive events (emergencies, holidays)?"**
   - Real-world pricing isn't a pure optimization — there are political and ethical constraints.

5. **"How do drivers currently respond to surge? What's the supply-side elasticity?"**
   - If drivers take 15 minutes to respond to surge, real-time pricing is less useful than forward-looking pricing.

---

## Phase 2: Problem Framing

### The Core Tension (State This Explicitly)

> "This is a **two-sided marketplace pricing problem** with a temporal dimension. We're not just setting a price — we're designing a *mechanism* that simultaneously:
> 1. Allocates scarce supply (drivers) to heterogeneous demand (riders with different willingness-to-pay)
> 2. Incentivizes supply to relocate (geographic and temporal signaling)
> 3. Manages user expectations and perceived fairness
> 4. Operates in real-time with noisy, incomplete information"

### The Fundamental Question:

> "At its core, we're asking: **what is the market-clearing price at every point in space and time, and how close can we get to it without making users feel exploited?**"

---

## Phase 3: Decomposition

### Sub-Problem A: Demand Forecasting

We need to know demand *before* it fully materializes:

- **Short-term (next 10-30 min)**: Event schedules, time of day, weather, historical patterns
- **Real-time signals**: App opens without booking, search queries, nearby events ending

### Sub-Problem B: Supply Forecasting

- **Current supply**: Drivers online, en-route (ETA to availability), headed toward the zone
- **Latent supply**: Offline drivers near the zone who *would* come online at a certain price point
- **Supply response time**: The lag between price signal and actual supply increase

### Sub-Problem C: Price Optimization

Given forecasted supply $$S(p)$$ and demand $$D(p)$$, find price $$p^*$$ such that:

$$p^* = \arg\min_p \; L(p)$$

Where the loss function $$L(p)$$ might be:

$$L(p) = w_1 \cdot \text{E}[\text{WaitTime}(p)] + w_2 \cdot \text{ChurnProbability}(p) - w_3 \cdot \text{Revenue}(p) + w_4 \cdot \text{Volatility}(p)$$

The weights $$w_1, w_2, w_3, w_4$$ encode the business's priorities.

### Sub-Problem D: Fairness & Communication

- How do you explain the price to the user?
- How do you prevent price discrimination (same ride, different prices for different users) from becoming a PR crisis?
- Should there be a cap? Should the price be "smooth" (no sudden jumps)?

---

## Phase 4: Proposed Approaches

### Approach 1: Market-Clearing Price with Smoothing

**Idea**: Estimate demand curve $$D(p)$$ and supply curve $$S(p)$$ in real-time. Set price where they intersect, with a smoothing constraint.

$$p_t = \alpha \cdot p_{\text{market-clearing}} + (1 - \alpha) \cdot p_{t-1}$$

The smoothing parameter $$\alpha$$ prevents jarring price jumps.

**Pro**: Economically principled. 
**Con**: Requires accurate real-time elasticity estimates. Sensitive to model misspecification.

### Approach 2: Multi-Armed Bandit / Contextual Pricing

**Idea**: Treat price-setting as an exploration-exploitation problem. For each context (location, time, weather, event), learn the optimal price through controlled experimentation.

**Pro**: Learns ground truth rather than relying on parametric demand models. 
**Con**: Exploration is expensive (you're charging some users "wrong" prices to learn). Ethical concerns.

### Approach 3: Predictive Surge (Forward-Looking)

**Idea**: Instead of reacting to current imbalance, **predict** imbalance 15-30 minutes ahead and pre-position supply.

- If a concert ends at 10pm, start signaling drivers at 9:45pm
- Price rises *gently* before the demand spike rather than *sharply* during it

**Pro**: Reduces peak surge magnitude. Better driver experience (less reactive driving). 
**Con**: Requires strong event detection and demand forecasting. False positives waste driver time.

---

## Phase 5: Deep-Dive on Approach 3 (Predictive Surge)

### The Forecasting Model

**Features**:
- Temporal: hour of day, day of week, holiday indicator
- Spatial: geohash, proximity to venues/transit hubs
- Contextual: weather, nearby event schedules (ticketing APIs), historical demand at this location+time
- Real-time signals: rate of app-opens in the area, rate of incoming requests in adjacent zones

**Target**: $$\hat{D}_{t+\Delta}(\text{zone})$$ = predicted ride requests in zone $$z$$ at time $$t + \Delta$$

**Model choice**: Gradient-boosted trees (XGBoost/LightGBM) for interpretability and speed. Not deep learning — the feature space is tabular and latency matters.

### The Pricing Logic

$$\text{surge}_t = \max\left(1.0, \; \frac{\hat{D}_{t+\Delta}}{\hat{S}_{t+\Delta}} \cdot \gamma\right)$$

Where $$\gamma$$ is a calibration constant tuned to keep average surge at a target level.

But apply **smoothing** and **caps**:
- Max surge change per 5-min window: 0.3x
- Hard cap at 3.0x (regulatory/PR)
- Minimum 10-minute duration at any surge level (prevents flickering)

### Driver Signaling Strategy

> "The key insight is that surge price is *both* a demand-rationing mechanism AND a supply-signaling mechanism. We can decouple these."

- Show drivers a **heat map** of predicted demand 15-30 min out
- Offer **guaranteed minimum earnings** for repositioning ("drive to zone X in next 10 min, guaranteed $$Y for your next ride")
- This reduces the need for high surge multipliers

---

## Phase 6: Validation & Experimentation

### How to Test This

**Geo-based A/B test** (switchback design):
- Randomly assign city regions to treatment (predictive surge) vs. control (reactive surge)
- Switch assignments every few hours to handle time-of-day effects
- Measure: average wait time, ride completion rate, driver utilization, rider NPS

**Key metric decomposition**:
- **Efficiency**: Rides completed per driver-hour
- **Experience**: Rider wait time, surge magnitude distribution
- **Revenue**: Revenue per ride, rides per hour
- **Retention**: 7-day rider return rate, 7-day driver online rate

### Known Failure Modes to Monitor:
- **Over-smoothing**: Surge doesn't respond fast enough to genuine spikes → long wait times
- **Under-smoothing**: Users experience confusing rapid price changes
- **Demand leakage**: Users learn to wait for surge to drop, creating artificial demand waves
- **Supply gaming**: Drivers learn the prediction pattern and position themselves to exploit guaranteed minimums

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Frames as supply-demand matching | Recognizes the two-sided nature: price is BOTH rationing AND signaling |
| Proposes a demand forecast model | Discusses the supply-side response lag and why reactive pricing is fundamentally limited |
| Suggests an A/B test | Proposes switchback design (standard A/B doesn't work for marketplace effects due to spillovers) |
| Optimizes for a single metric | Explicitly states that objectives conflict and proposes a weighted loss function with stakeholder-chosen weights |

# Case Study 3: Meta — "Detect Coordinated Inauthentic Behavior at Scale"

---

## The Problem Statement (As Given by the Interviewer)

> *"We're seeing networks of fake accounts that are sophisticated enough to evade our existing classifiers. They have real-looking profiles, post organic-seeming content, and only occasionally engage in coordinated behavior (mass reporting, amplification campaigns, scam distribution). How would you detect them?"*

---

## Phase 1: Clarifying Questions

### The Questions That Reveal Systems Thinking:

1. **"What does 'coordinated' mean operationally? Are we looking for accounts that act together, or accounts that are individually fake but happen to be in a network?"**
   - *Why critical*: An individually fake account (single bot) is a classification problem. A coordinated network is a graph problem. The methods are fundamentally different.

2. **"What's the cost asymmetry? What's worse — banning a real user or missing a fake one?"**
   - If we're talking about state-sponsored disinformation: missing is much worse (national security).
   - If we're talking about spam farms: false positives are worse (losing real users).
   - This determines our threshold and whether we auto-ban or flag for human review.

3. **"What's our current detection pipeline? What features do existing classifiers use, and where are they failing?"**
   - If current models use profile features (age, photo, bio), the adversary has likely already learned to bypass those. We need to look at *behavioral* and *relational* features.

4. **"What time horizon do we operate on? Real-time (block at sign-up) vs. retrospective (find established fakes)?"**
   - Different cadences need different approaches. Real-time has latency constraints. Retrospective can use the full graph.

5. **"Do we have labeled data? How was it collected? What's the base rate of fake accounts?"**
   - If base rate is 0.1%, even a 99% accurate classifier produces more false positives than true positives. This is the class imbalance problem.

---

## Phase 2: Problem Framing

### Restating the Problem

> "We are looking for **clusters of accounts that exhibit temporal correlation in their actions** — they act in ways that are individually normal but collectively improbable without coordination. The key signal isn't what they do, it's **when they do it together**."

### The Fundamental Insight:

> "Individual-level classifiers will always be in an arms race with adversaries. But **coordination is hard to fake** — to be effective, a network must act together, and that temporal/structural correlation is the exploitable signal. The adversary faces a fundamental tradeoff: the more coordinated their behavior (effectiveness), the more detectable they become."

---

## Phase 3: Decomposition

### Sub-Problem A: Define "Coordination" Mathematically

Two accounts $$i$$ and $$j$$ are suspiciously coordinated if:

$$P(\text{action}_i \text{ and } \text{action}_j \text{ within } \Delta t) \gg P(\text{action}_i) \cdot P(\text{action}_j)$$

The ratio of joint probability to independent probability is the **coordination score**:

$$\text{CoordScore}(i,j) = \frac{P(A_i \cap A_j | \Delta t < \tau)}{P(A_i) \cdot P(A_j)}$$

Actions to measure: posting, liking, commenting, reporting, friend-requesting the same targets.

### Sub-Problem B: Graph Construction

Build a **behavioral similarity graph**:
- **Nodes** = accounts
- **Edges** = high coordination score OR shared behavioral fingerprint
- **Edge weight** = strength of coordination evidence

Then look for **dense subgraphs** (communities) that are anomalously tightly connected.

### Sub-Problem C: Distinguish Coordination from Organic Communities

> "A real fan club for a K-pop group will ALSO show coordinated liking behavior. How do we distinguish malicious coordination from organic communities?"

Key differentiators:
- **Account age distribution**: Real communities have diverse join dates. Fake networks often have clustered creation dates.
- **Content originality**: Real users create diverse original content. Fake networks amplify identical or near-identical content.
- **Behavioral diversity**: Real users have varied activity patterns. Fake accounts often have suspiciously similar online/offline cycles.
- **Target diversity**: Real communities engage with many targets. Fake networks focus on specific manipulation targets.

### Sub-Problem D: Scale

With $$\sim$$3 billion accounts, pairwise coordination scores are $$O(n^2)$$ — infeasible.

Need **candidate generation** before scoring:
- Accounts that engaged with the same content within short windows
- Accounts created from same IP/device fingerprint clusters
- Accounts with similar behavioral fingerprints (posting times, session lengths)

---

## Phase 4: Proposed Approaches

### Approach 1: Temporal Co-Occurrence Mining

**Idea**: For each "suspicious action" (mass reporting, amplification of same content), identify accounts that participated within a tight time window. Build a co-occurrence matrix. Apply community detection.

**Algorithm**:
1. For each target entity (post, page, user) that received unusual engagement:
   - Extract all accounts that engaged within $$\tau$$ seconds of each other
   - Build bipartite graph: accounts → targets
2. Project to account-account graph (shared targets)
3. Apply Louvain/Leiden community detection
4. Score communities by: density, account age variance, content diversity

**Pro**: Directly targets the coordination signal. 
**Con**: Only catches accounts *during* coordinated actions. Misses sleeper accounts.

### Approach 2: Graph Neural Network on Social Graph

**Idea**: Learn node embeddings that capture both individual features AND neighborhood structure. Accounts in fake networks will cluster in embedding space.

$$h_v^{(k)} = \sigma\left(W^{(k)} \cdot \text{AGGREGATE}\left(\{h_u^{(k-1)} : u \in \mathcal{N}(v)\}\right)\right)$$

**Pro**: Can learn complex structural patterns. Generalizes to unseen network structures. 
**Con**: Expensive to train at Meta's scale. Requires labeled networks for supervised training. May not explain *why* a network is flagged.

### Approach 3: Behavioral Fingerprinting + Anomaly Detection

**Idea**: Create a high-dimensional behavioral fingerprint for each account (posting cadence, session patterns, content engagement patterns). Accounts controlled by the same operator will have similar fingerprints (same automation tools, same human operator patterns).

**Pro**: Catches sleeper accounts before they act. Doesn't require the accounts to coordinate. 
**Con**: Higher false positive rate (legitimate users with similar habits). Needs careful feature engineering.

---

## Phase 5: Deep-Dive on Approach 1 + Hybrid

### The Ideal Candidate Proposes a Staged System:

**Stage 1 — Candidate Generation (Fast, High Recall)**:
- Trigger on anomalous engagement patterns on any entity
- Pull all accounts involved in the engagement burst
- Use simple heuristics: accounts created within 7 days of each other, similar usernames, shared device fingerprints

**Stage 2 — Network Scoring (Slower, High Precision)**:
- For candidate clusters, compute full behavioral similarity matrix
- Features: temporal action correlation, content overlap, target overlap, communication patterns within cluster
- Apply a trained classifier on cluster-level features

**Stage 3 — Human Review + Action**:
- High-confidence clusters: auto-restrict (shadow-ban, reduce distribution)
- Medium-confidence: route to human review with explanation
- Low-confidence: add to watchlist, re-score weekly

### The Adversarial Consideration:

> "Any signal we use, the adversary will eventually adapt to. So the system needs to be **modular** — we can swap in new features/signals without redesigning the pipeline. And we should maintain **feature secrecy** — don't publish which signals we use (unlike spam filters, which can be transparent)."

---

## Phase 6: Validation

### Metrics for a Detection System:

| Metric | Definition | Target |
| --- | --- | --- |
| Precision @ human review | % of flagged networks that reviewers confirm as fake | > 80% |
| Recall (estimated) | % of known fake networks caught | > 60% |
| Time-to-detection | Hours from first coordinated action to flagging | < 24h |
| Adversarial robustness | How quickly detection degrades after adversary adapts | Measure monthly |

### Key Validation Strategies:
- **Red team exercises**: Internal team creates fake networks and tests if system catches them
- **Temporal holdout**: Train on networks discovered in month M, evaluate on month M+1
- **Cross-campaign generalization**: Train on one type of campaign (spam), test on another (disinformation)

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes a binary classifier for fake accounts | Recognizes this is a *network* detection problem, not an individual classification problem |
| Uses profile features | Argues that coordination timing is the hardest signal for adversaries to eliminate |
| Builds one model | Proposes a staged pipeline (fast candidate generation → precise scoring → human review) |
| Evaluates with accuracy | Discusses precision-recall tradeoff in context of cost asymmetry AND base rate |
| Treats it as a static problem | Explicitly discusses the adversarial nature and need for modularity |

# Case Study 4: Airbnb — "Redesign Search Ranking to Maximize Long-Term Marketplace Health"

---

## The Problem Statement (As Given by the Interviewer)

> *"Airbnb's search ranking currently optimizes for booking probability. But we've noticed that top-ranked listings get booked immediately, mid-tier listings struggle, and new hosts never get visibility. The marketplace is becoming winner-take-all. How would you redesign ranking to balance short-term conversions with long-term marketplace health?"*

---

## Phase 1: Clarifying Questions

### Questions That Reveal Marketplace Thinking:

1. **"How do we define 'marketplace health'? Is it supply growth, supply diversity, geographic coverage, price distribution, or host retention?"**
   - *Why critical*: "Marketplace health" is not a single metric. The candidate must push for specificity.

2. **"What's the current host churn rate, and is it correlated with listing visibility?"**
   - If hosts who get zero bookings in their first month churn at 80%, then giving new hosts visibility has quantifiable long-term revenue value.

3. **"Are there quality signals we can use to separate 'good listings that lack exposure' from 'listings that rank low because they're genuinely poor'?"**
   - This is the core challenge: we want to boost *deserving* under-exposed listings, not just any under-exposed listing.

4. **"What's the guest experience cost of showing a sub-optimal result? Do guests book and then cancel? Leave bad reviews? Or just scroll past?"**
   - The cost of experimentation (showing non-optimal results) determines how aggressively we can explore.

5. **"Is there a seasonality or geographic component? Are some markets more winner-take-all than others?"**
   - Dense urban markets (Paris, NYC) may need different strategies than sparse markets.

---

## Phase 2: Problem Framing

### The Core Tension (Stated Explicitly)

> "This is an **exploration-exploitation tradeoff in a two-sided marketplace**. Every search result we show is simultaneously:
> 1. A **service to the guest** (show them the best listing)
> 2. A **signal to the host** (visibility = motivation to stay on platform)
> 3. An **investment in the marketplace** (exposure for new/uncertain listings generates information)

> Optimizing purely for (1) starves (2) and (3), creating a death spiral where supply concentrates and eventually guests have fewer choices."

### The Economic Framing:

> "Every impression we give to a new listing has a **long-term option value** — it generates reviews, which reduce uncertainty, which either confirms the listing is good (future bookings without boost) or reveals it's bad (we stop showing it). This information has computable value."

---

## Phase 3: Decomposition

### Sub-Problem A: Quantify the Value of Information

For a new listing with $$n$$ reviews, the uncertainty about its quality is:

$$\sigma(q_i) \propto \frac{1}{\sqrt{n_i + 1}}$$

The **value of one more impression** for listing $$i$$ is:

$$\text{VOI}_i = \text{E}[\Delta \text{Revenue}_i | \text{one more review}] \cdot P(\text{book} | \text{impression})$$

Listings with high uncertainty AND high prior quality signal have the highest VOI.

### Sub-Problem B: Define the Multi-Objective

We want to maximize a blended objective:

$$\text{Objective} = \underbrace{\sum_i P(\text{book}_i) \cdot \text{BookingValue}_i}_{\text{Short-term revenue}} + \lambda \cdot \underbrace{\sum_i \text{VOI}_i \cdot \text{Impression}_i}_{\text{Long-term marketplace value}}$$

The parameter $$\lambda$$ controls the exploration-exploitation tradeoff. Too high = bad guest experience. Too low = marketplace concentration.

### Sub-Problem C: Constrained Optimization

Rather than blending into a single score, we might impose **constraints**:
- At least 20% of impressions go to listings with < 5 reviews (new host exposure guarantee)
- No single listing receives > X% of impressions in its market (anti-concentration)
- Guest satisfaction (measured by post-booking NPS) doesn't drop below threshold

This becomes a constrained ranking problem, solvable via linear programming at serve time.

### Sub-Problem D: Cold Start for New Listings

A new listing has no reviews, limited photos, maybe an incomplete description. How do we estimate its quality?

**Prior quality signals**:
- Host's performance on other listings (if any)
- Listing attributes vs. similar high-performing listings (price positioning, amenities, photos)
- Host responsiveness during onboarding (response rate to inquiries)
- Market-level base rates (what's the average new listing performance in this area?)

---

## Phase 4: Proposed Approaches

### Approach 1: Thompson Sampling for Ranking

**Idea**: Model each listing's booking probability as a Beta distribution. For new listings, the prior is wide (uncertain). Each impression updates the posterior. At ranking time, *sample* from each listing's distribution and rank by the sample.

$$\text{score}_i \sim \text{Beta}(\alpha_i + \text{successes}_i, \; \beta_i + \text{failures}_i)$$

**Pro**: Mathematically principled exploration. Automatically reduces exploration for listings we've learned about. 
**Con**: Doesn't account for booking value (only probability). May explore too aggressively for high-stakes searches (e.g., honeymoon trips).

### Approach 2: Position-Aware Auction with Boost Budget

**Idea**: Each new listing gets a "boost budget" (virtual impressions). The budget is spent as a bid in a position auction. Once depleted, the listing competes on pure quality.

$$\text{EffectiveScore}_i = \text{QualityScore}_i + \text{BoostBid}_i \cdot \mathbf{1}(\text{budget}_i > 0)$$

**Pro**: Simple, transparent, controllable. Hosts can understand the system. 
**Con**: Doesn't adapt to listing quality. A bad listing burns its entire budget before we learn it's bad.

### Approach 3: Counterfactual-Aware Ranking

**Idea**: For each position in the search results, estimate the *marginal* value of placing listing $$i$$ there vs. the next-best alternative. Rank by marginal value, where value includes long-term marketplace effects.

$$\text{MarginalValue}_i^{(k)} = \text{ImmediateValue}_i^{(k)} + \lambda \cdot \text{FutureLTV}_{\text{host}_i}(\text{extra impression})$$

Where $$\text{FutureLTV}$$ is the expected lifetime value of the host given one more positive interaction.

**Pro**: Directly optimizes for what we care about (total marketplace value over time). 
**Con**: Requires estimating host LTV as a function of early experiences. Complex to implement.

---

## Phase 5: Deep-Dive on Approach 3 (Counterfactual-Aware)

### Estimating Host LTV

Build a survival model for hosts:

$$P(\text{host churns} | \text{bookings in first 60 days} = k) = f(k, \text{market}, \text{host type})$$

From historical data, we can estimate:
- A new host with 0 bookings in 60 days has 70% churn probability
- A new host with 1 booking has 40% churn probability  
- A new host with 3+ bookings has 10% churn probability

The marginal value of the *first* booking for a new host is enormous:

$$\Delta \text{LTV} = (0.70 - 0.40) \times \text{AverageHostLTV} = 0.30 \times \text{AverageHostLTV}$$

If average host LTV is \$5,000 over their lifetime, the first booking is worth $$\$1,500$$ in expected marketplace value.

### Position-Dependent Click Probability

The value of an impression depends on position:

$$P(\text{click} | \text{position} = k) \approx \frac{1}{k^{0.7}}$$  (empirical power law)

So position 1 is roughly 2.5x more valuable than position 5. We don't need to put new listings in position 1 — positions 4-8 still provide meaningful exposure with minimal guest experience cost.

### The Algorithm:

1. Score all candidate listings by $$P(\text{book}) \times \text{BookingValue}$$
2. For each position $$k = 1, ..., K$$:
   - Compute opportunity cost of *not* placing the top-scoring listing there
   - Compute long-term value of placing an underexposed listing there
   - If long-term value > opportunity cost: place underexposed listing
   - Otherwise: place top-scoring listing
3. Constraint: maximum 2-3 "boosted" slots per search results page

---

## Phase 6: Validation & Guardrails

### Experimentation Design

**Why standard A/B testing is tricky here**:
- Two-sided marketplace → treatment on one side affects the other
- Long-term effects → need long experiment duration (weeks, not days)
- Network effects → interference between treatment and control

**Better design**: Geo-randomized experiment
- Randomly assign *markets* (cities) to treatment vs. control
- Run for 3-6 months to capture long-term supply effects
- Primary metric: total bookings per market (captures both sides)
- Secondary: host activation rate, new host 90-day retention, guest rebooking rate

### Guardrails (Hard Stops):
- Guest-side: If booking conversion drops > 3% → reduce $$\lambda$$
- Host-side: If bad-review rate on boosted listings > 2x average → improve cold-start quality estimation
- System: If any single listing receives boost and maintains < 2% booking rate after 50 impressions → remove from boost pool

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes boosting new listings | Quantifies the *value* of exploration and frames it as an investment with computable ROI |
| Acknowledges the tradeoff exists | Derives the specific tradeoff as a constrained optimization with economic justification |
| Suggests an A/B test | Recognizes why standard A/B fails in marketplaces and proposes geo-randomization |
| Mentions cold start | Builds a full cold-start quality estimation framework using transferable signals |
| Treats ranking as a single-objective problem | Explicitly decomposes into guest value, host value, and marketplace value with separate estimation for each |

# Case Study 5: Spotify — "We Launched a Feature and Metrics Went Up. Should We Ship It?"

---

## The Problem Statement (As Given by the Interviewer)

> *"Spotify launched a new feature: 'AI DJ' — a personalized radio-style experience with AI commentary between songs. In our A/B test, treatment users showed +5% listening time and +3% DAU. But some team members are concerned the results are misleading. Your job is to determine whether we should ship this to 100% of users. What would you investigate?"*

---

## Phase 1: Clarifying Questions

### Questions That Signal Statistical & Business Maturity:

1. **"How long has the experiment been running, and what's the sample size?"**
   - A +5% lift that's been running for 3 days might be novelty effect. The same lift after 6 weeks is much more credible.

2. **"What's the randomization unit? User-level or session-level?"**
   - User-level: clean, standard. Session-level: potential issues with users crossing between treatment and control.

3. **"Was there a pre-registration of primary metrics and hypothesis?"**
   - If the team looked at 20 metrics and found 2 that moved... that's not the same as pre-specifying "listening time" and finding it moved.

4. **"What's the variance of the metric? Is +5% listening time 0.5 standard deviations or 0.05?"**
   - A lift can be statistically significant but practically meaningless, or vice versa.

5. **"Are the concerned team members worried about a specific mechanism (cannibalization, novelty, subgroup harm) or just general skepticism?"**
   - This focuses the investigation.

6. **"What's the business model implication? Does more listening time translate to more revenue, or could it shift revenue-generating behavior (e.g., users listen to AI DJ instead of engaging with ads/playlists that drive paid conversions)?"**

---

## Phase 2: Problem Framing

### Restating the Problem

> "The question isn't 'is the metric lift real?' — it's 'is the metric lift **durable**, **causal**, **not offset by hidden costs**, and **reflective of genuine value creation rather than value redistribution**?'"

### The Hierarchy of Threats to Validity:

> An ideal candidate structures their investigation as a hierarchy:
>
> **Level 1: Is the experiment mechanically sound?** (Randomization, metric definition, statistical test)
> 
> **Level 2: Is the effect real and durable?** (Novelty, primacy, time-varying treatment effects)
>
> **Level 3: Is the effect what it appears to be?** (Metric gaming, cannibalization, composition effects)
>
> **Level 4: Is the effect net-positive for the business?** (Ecosystem effects, long-term implications, hidden costs)

---

## Phase 3: Decomposition — The Investigation Checklist

### Level 1: Experimental Validity

#### Check 1.1: Randomization Integrity (Sample Ratio Mismatch)

Compute the ratio of users in treatment vs. control. If it deviates significantly from 50/50:

$$\chi^2 = \frac{(n_T - n_C)^2}{n_T + n_C}$$

If $$\chi^2 > 3.84$$ ($$p < 0.05$$), there's a **Sample Ratio Mismatch (SRM)** — the experiment is mechanically broken.

Common causes: triggered experiment only fires for users with certain app versions, treatment causes more crashes (users exit and re-randomize), bot filtering removes users asymmetrically.

#### Check 1.2: Pre-Treatment Balance (AA Check)

Compare treatment and control on pre-experiment metrics (last 30 days of listening before experiment start). They should be statistically identical. Any significant difference indicates broken randomization.

#### Check 1.3: Multiple Testing Correction

If 10 metrics were examined, apply Benjamini-Hochberg correction:

$$\text{Adjusted } p_i = p_i \cdot \frac{m}{\text{rank}(p_i)}$$

Where $$m$$ = number of tests. The +5% and +3% may not survive correction.

---

### Level 2: Effect Durability

#### Check 2.1: Novelty Effect

> "Users try new things because they're new, not because they're good."

Plot treatment effect **over time** (by week of experiment):

$$\hat{\tau}_w = \bar{Y}_{T,w} - \bar{Y}_{C,w} \quad \text{for } w = 1, 2, ..., W$$

- If $$\hat{\tau}_w$$ is decreasing over time → novelty effect. Don't ship.
- If $$\hat{\tau}_w$$ is stable or increasing → genuine adoption. Good signal.
- If $$\hat{\tau}_w$$ spikes then stabilizes at a lower (but positive) level → partial novelty. Adjust expectations.

#### Check 2.2: Cohort-Specific Effects (Primacy Bias)

Existing users have established habits. New users have no anchor. Break down:
- Users who joined before experiment: do they sustain engagement after initial exploration?
- Users who joined during experiment: do they engage more because AI DJ is their "normal" (primacy bias)?

#### Check 2.3: Day-of-Week and Context Effects

Is the lift uniform across:
- Weekday commute vs. weekend leisure?
- Mobile vs. desktop?
- Free tier vs. premium?

If the lift is concentrated in one context, the aggregate number overstates the general effect.

---

### Level 3: What the Effect Actually Represents

#### Check 3.1: Cannibalization

> "Did AI DJ *create* new listening, or did it *steal* listening from other features?"

Decompose total listening time:

$$\Delta \text{Total} = \Delta \text{AI DJ} + \Delta \text{Playlists} + \Delta \text{Albums} + \Delta \text{Podcasts} + \Delta \text{Search/Explore}$$

If $$\Delta \text{Total} = +5\%$$ but $$\Delta \text{Playlists} = -4\%$$ and $$\Delta \text{AI DJ} = +9\%$$, the *net* creation is only +5%, and we've cannibalized playlists.

**Why this matters**: If playlists drive better ad targeting (more intent signal) or better discovery (long-term retention), cannibalizing them for AI DJ time might be net-negative despite the aggregate lift.

#### Check 3.2: Composition Effects (Simpson's Paradox)

Is the +5% driven by:
- **Intensive margin**: Same users listening more? (Good)
- **Extensive margin**: More users becoming active, but each listens less? (Might be fine)
- **Compositional shift**: Heavy users engaging more, light users engaging less? (Concerning — we're concentrating on power users)

Compute:

$$\Delta \text{Total} = \underbrace{\Delta(\text{DAU}) \times \overline{\text{Time/User}}_{\text{pre}}}_{\text{Extensive margin}} + \underbrace{\text{DAU}_{\text{pre}} \times \Delta(\overline{\text{Time/User}})}_{\text{Intensive margin}} + \underbrace{\text{interaction term}}_{\text{usually small}}$$

#### Check 3.3: Engagement Quality vs. Quantity

> "More time listened doesn't mean more value if users are passively zoning out vs. actively enjoying."

Check secondary quality signals:
- Skip rate during AI DJ vs. user-curated playlists
- Song saves/likes during AI DJ sessions
- Explicit feedback (thumbs up/down)
- Session start patterns (are users actively choosing AI DJ or just landing there by default?)

---

### Level 4: Business & Ecosystem Effects

#### Check 4.1: Revenue Impact

Listening time ≠ revenue. Check:
- **Ad-supported users**: More listening = more ad impressions? Or does AI DJ format reduce ad slots?
- **Premium conversion**: Does AI DJ increase or decrease free-to-premium conversion? (If free experience is "too good," conversion might drop)
- **Artist/label economics**: Does AI DJ change royalty distribution patterns in ways that could create licensing issues?

#### Check 4.2: Content Creator Ecosystem

If AI DJ disproportionately plays popular tracks (safe recommendations), it might:
- Reduce exposure for emerging artists
- Concentrate streams on top artists (bad for marketplace diversity)
- Create long-term content supply issues (artists leave platform)

#### Check 4.3: Long-Term Retention Signal

> "The ultimate metric isn't this week's listening time — it's whether users are still here in 6 months."

A/B tests are often too short to capture retention effects. Estimate using:
- **Leading indicators**: playlist creation, social sharing, profile customization (investment behaviors)
- **Historical analogy**: When similar features launched previously, what was the long-run retention delta?

---

## Phase 4: The Decision Framework

### After All Checks, Structure the Recommendation:

$$\text{Ship Decision} = f(\text{Effect Durability}, \text{Net Value Creation}, \text{Ecosystem Health}, \text{Reversibility})$$

| Scenario | Evidence Pattern | Recommendation |
| --- | --- | --- |
| Clean win | Stable effect over time, net positive creation, no cannibalization, quality metrics up | Ship to 100% |
| Novelty-inflated | Declining treatment effect over time | Wait 2 more weeks, re-evaluate |
| Cannibalization | Aggregate up, but displaces higher-value features | Ship with guardrails (cap AI DJ usage? interleave with playlists?) |
| Subgroup harm | Overall positive, but specific user segments worse off | Ship with targeting (exclude harmed segments, iterate on their experience) |
| Metric gaming | Listening time up but quality signals down | Do NOT ship. Redefine success metric. |

---

## Phase 5: What the Ideal Candidate Communicates

> "My recommendation is [ship / don't ship / ship with conditions], and here's my confidence level.
> 
> The +5% listening time lift is [real/partially inflated by novelty/confounded by cannibalization].
> 
> The key risk is [X], which I'd mitigate by [Y].
> 
> If we ship, I'd monitor [Z] over the next 8 weeks with a kill-switch if [threshold] is breached."

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Checks for statistical significance | Checks SRM, multiple testing, AND effect stability over time |
| Says "it might be novelty" | Proposes specific diagnostic (plot treatment effect by experiment week, quantify decay rate) |
| Reports the aggregate effect | Decomposes into extensive/intensive margin and checks for Simpson's paradox |
| Trusts the primary metric | Questions whether the primary metric actually maps to business value |
| Gives a binary ship/no-ship | Provides a decision matrix with conditions, guardrails, and monitoring plan |

---
---

# PART II: ML System Design Case Studies

---

## Why ML System Design Is Different

ML system design interviews at top companies (Google, Meta, Amazon, YouTube, LinkedIn) are **not** about building the best model. They're about designing a **production system** that:

1. **Solves the right problem** — Not all business problems need ML. The first question is always: "Is ML the right tool here?"
2. **Operates under real constraints** — Latency, cost, data freshness, cold start, privacy, fairness
3. **Evolves gracefully** — Distribution shift, concept drift, adversarial adaptation, scaling
4. **Fails safely** — What happens when the model is wrong? What's the fallback?

---

## The ML System Design Framework

| Phase | What You Do | Common Failure Mode |
| --- | --- | --- |
| **1. Problem Setup** | Define task type, identify what "success" means in production (not just offline metrics) | Jumping to model architecture |
| **2. Data** | What data exists? What's missing? How is it labeled? How does it drift? | Assuming clean, abundant, static data |
| **3. Feature Engineering** | What signals predict the outcome? What's available at serving time? | Using features unavailable at inference |
| **4. Model** | Architecture choices, training strategy, offline evaluation | Over-indexing on model complexity |
| **5. Serving** | Latency requirements, batch vs. real-time, caching strategies | Ignoring production constraints |
| **6. Monitoring & Iteration** | How to detect degradation, retrain, handle feedback loops | Treating deployment as the finish line |

*The following 5 case studies exercise this framework on real ML system design problems.*

# Case Study 6: Google Pay — "Design a Real-Time Payment Fraud Detection System"

---

## The Problem Statement (As Given by the Interviewer)

> *"Google Pay processes millions of transactions daily. We need a system that can flag fraudulent transactions in real-time (under 100ms) while keeping false positive rates low enough that legitimate users aren't constantly blocked. Design this end-to-end."*

---

## Phase 1: Clarifying Questions

### Questions That Reveal Production ML Maturity:

1. **"What types of fraud are we detecting? Account takeover? Stolen credentials? Social engineering? Money laundering? Friendly fraud (chargebacks)?"**
   - Each fraud type has different signals, timescales, and base rates. A single model rarely handles all types well.

2. **"What's the current fraud rate and what's the dollar-weighted false negative cost vs. false positive cost?"**
   - If fraud rate is 0.1% and average fraud is \$500 but blocking a legitimate \$5 coffee costs \$0.50 in user friction... the cost matrix is wildly asymmetric.
   - $$\text{Expected Cost} = \text{FN\_rate} \times \text{Avg\_fraud\_amount} + \text{FP\_rate} \times \text{Friction\_cost\_per\_block}$$

3. **"What latency budget do we have? Is 100ms the hard ceiling, or can we have a two-tier system (fast decision + async review)?"**
   - This fundamentally shapes architecture. 100ms means no complex graph features at serve time.

4. **"What data is available at transaction time vs. only after the fact?"**
   - Transaction amount, merchant, device, location → available immediately
   - Whether the user disputes the charge → available days/weeks later (label delay)

5. **"How do fraudsters adapt? What's the typical lifecycle of a fraud pattern before it changes?"**
   - If patterns shift weekly, monthly retraining is too slow. Need online learning or fast retraining.

---

## Phase 2: Problem Framing

### The System-Level View

> "This is not just a classification problem. It's a **decision system** that must:
> 1. Score transactions in real-time under strict latency
> 2. Handle extreme class imbalance ($$\sim$$0.1% positive rate)
> 3. Adapt to adversarial distribution shift (fraudsters evolve)
> 4. Manage the precision-recall tradeoff dynamically based on business context
> 5. Provide explainability for blocked transactions (user trust + regulatory compliance)"

### The Key Architectural Insight:

> "We need a **multi-stage system**, not a single model. Fast, cheap filters handle the easy cases. Expensive, accurate models handle the ambiguous ones. Human review handles the hardest cases."

---

## Phase 3: System Architecture Decomposition

### Stage 1: Rule Engine (< 5ms)

**Purpose**: Catch obvious fraud and obvious legitimate transactions instantly.

- Hard rules: transaction from sanctioned country, known fraudulent device fingerprint, card reported stolen
- Whitelist rules: recurring subscription to same merchant, small transaction from home location
- Pass-through: ~70% of transactions are trivially legitimate and skip the ML model entirely

**Why rules first**: Rules are fast, explainable, auditable, and don't need retraining. They handle the distribution tails.

### Stage 2: Real-Time ML Scoring (< 50ms)

**Purpose**: Score the ambiguous 30% that rules can't resolve.

**Feature Categories**:

| Category | Examples | Availability |
| --- | --- | --- |
| Transaction | Amount, merchant category, time of day, currency | Immediate |
| User profile | Account age, avg transaction size, typical merchants | Pre-computed |
| Velocity | Transactions in last 1h/24h/7d, unique merchants in last 24h | Near-real-time aggregates |
| Device/session | Device fingerprint, IP geolocation, session duration | Immediate |
| Graph (pre-computed) | Degree centrality in transaction graph, shared device with known fraudster | Batch-updated |

**Critical Design Decision**: Which features can be computed within 50ms?

- Pre-compute user profiles and graph features in batch (hourly/daily)
- Maintain real-time velocity counters in a streaming system (Flink/Kafka)
- Transaction-level features are immediate

### Stage 3: Async Deep Analysis (< 30 minutes)

**Purpose**: For medium-confidence scores, run expensive analysis before finalizing.

- Full graph traversal: is this account connected to known fraud rings?
- Behavioral sequence modeling: does the session trajectory look like a bot?
- Cross-reference with other signals (email, phone number velocity)

**UX**: Place a hold on the transaction. Notify user "verification in progress." Release or block within 30 min.

### Stage 4: Human Review

**Purpose**: High-value, low-confidence cases get manual review.

- Provide analysts with model explanation, feature importance, similar historical cases
- Analyst decisions feed back as training labels

---

## Phase 4: The ML Model (Stage 2 Deep-Dive)

### Model Architecture Choices:

| Option | Pro | Con |
| --- | --- | --- |
| Gradient-boosted trees (XGBoost) | Fast inference, handles tabular data well, interpretable | Limited sequence modeling |
| Neural network (deep & wide) | Can learn complex interactions | Slower inference, less interpretable |
| Ensemble (trees + NN) | Best accuracy | Latency budget may not allow |

**Ideal choice for Stage 2**: XGBoost/LightGBM for the latency-critical path. Neural models can run in Stage 3.

### Training Strategy:

**The Label Problem**:
- Fraud labels arrive with **delay** (chargebacks come 30-90 days later)
- We can't wait 90 days to get labels for retraining

**Solution**: Multi-label strategy:
- **Confirmed fraud**: Chargebacks, user-reported unauthorized transactions (high precision, delayed)
- **Rules-triggered fraud**: Caught by existing rules (available immediately, but circular)
- **Analyst-labeled**: From human review queue (high quality, small volume)

Train on a blend, with sample weights reflecting label confidence:

$$\mathcal{L} = \sum_i w_i \cdot \ell(y_i, \hat{y}_i)$$

Where $$w_i$$ is higher for analyst-confirmed labels and lower for rule-triggered labels.

### Handling Class Imbalance:

- **Don't** oversample the minority class blindly (creates synthetic artifacts)
- **Do**: Use focal loss to down-weight easy negatives:

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- **Do**: Stratified sampling in training batches
- **Do**: Evaluate on precision-recall curves, not accuracy or AUC alone

### Threshold Selection:

The score threshold isn't fixed — it should depend on context:

$$\text{threshold}(x) = f(\text{transaction\_amount}, \text{user\_trust\_score}, \text{current\_fraud\_rate})$$

- High-value transactions: lower threshold (more cautious)
- Trusted, long-time users: higher threshold (less friction)
- During known fraud campaigns: lower threshold globally

---

## Phase 5: Feature Engineering Deep-Dive

### The Most Powerful Features (From Industry Experience):

**Velocity Features** (real-time counters):
- Number of transactions in last 1h, 6h, 24h, 7d
- Number of unique merchants in last 24h
- Amount spent in last 24h / average daily spend (ratio to baseline)
- Number of failed transactions in last 1h

**Deviation Features** (compare to user's own history):
$$z_i = \frac{x_i - \mu_{\text{user}}}{\sigma_{\text{user}}}$$

- How many standard deviations from the user's typical transaction amount?
- How far from the user's typical location?
- How different from the user's typical time-of-day?

**Interaction Features**:
- This merchant + this amount + this time → how common in population?
- This device + this location + new account → high risk signal

**Graph Features** (pre-computed):
- Shared device/IP with accounts that have been flagged
- Payment to merchant that receives from many flagged accounts
- Network distance to known fraud clusters

---

## Phase 6: Monitoring, Feedback Loops & Adversarial Adaptation

### The Feedback Loop Problem:

> "We only get labels for transactions we *allowed*. Transactions we blocked never get confirmed as fraud or not-fraud. This creates **selection bias** in our training data."

**Solution**: Periodically allow a small random sample of blocked transactions through (with enhanced monitoring) to get unbiased label estimates. This is the "exploration" tax.

### Monitoring Metrics:

| Metric | Frequency | Alert Threshold |
| --- | --- | --- |
| Model score distribution (PSI) | Hourly | PSI > 0.1 |
| False positive rate (from user appeals) | Daily | > 2x baseline |
| Fraud rate in approved transactions | Weekly (label delay) | > 1.5x baseline |
| Feature drift per feature | Daily | KS-stat > 0.05 |
| Latency p99 | Real-time | > 80ms |

### Adversarial Adaptation Strategy:

- **Model versioning**: Maintain last 3 model versions. If new model degrades, instant rollback.
- **Retraining cadence**: Weekly full retrain + daily incremental updates on new confirmed fraud.
- **Ensemble diversity**: Run 3 models trained on different time windows. If they disagree, escalate to Stage 3.
- **Feature rotation**: Periodically introduce new features and retire features that adversaries have learned to bypass.

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes a single ML model | Designs a multi-stage system (rules → fast ML → deep analysis → human review) |
| Lists features | Distinguishes features available at inference time from those that require pre-computation, and designs the data infrastructure accordingly |
| Mentions class imbalance | Addresses the label delay problem, selection bias from blocking, and proposes exploration strategies |
| Picks a model | Justifies the choice based on latency constraints, not just accuracy |
| Reports precision/recall | Proposes dynamic thresholds based on transaction context |

# Case Study 7: YouTube — "Design a Video Recommendation System That Maximizes Long-Term Engagement Without Creating Echo Chambers"

---

## The Problem Statement (As Given by the Interviewer)

> *"YouTube wants to redesign its recommendation system. The current system optimizes for watch time, but there are concerns that it: (1) creates filter bubbles, (2) promotes increasingly extreme content, and (3) favors clickbait over quality. Design a recommendation system that maximizes healthy, long-term user engagement."*

---

## Phase 1: Clarifying Questions

### Questions That Reveal ML Systems Maturity:

1. **"How do we define 'healthy engagement'? Is there an existing operational definition, or is this something we need to propose?"**
   - *Why critical*: "Watch time" is easy to measure. "Healthy engagement" is not. This might be the hardest part of the problem.

2. **"What's the serving context? Homepage recommendations? Up-next sidebar? Search results? Each has different constraints."**
   - Homepage: broad exploration, can tolerate more diversity
   - Up-next: continuation of a session, context-dependent
   - Search: intent-driven, relevance > exploration

3. **"What scale are we operating at? How many candidate videos exist? What's the latency budget per recommendation?"**
   - ~800M videos, serving recommendations in < 200ms. This means we CANNOT score all videos for every user — we need a funnel.

4. **"What explicit and implicit signals do we have about user satisfaction beyond watch time?"**
   - Likes/dislikes, shares, subscriptions after watching, survey responses, time-to-next-session (did they come back?), regret signals ("not interested" clicks)

5. **"Are there hard constraints? Content policies that must be enforced regardless of engagement?"**
   - Certain content categories (misinformation, violence, self-harm) must be demoted regardless of user engagement signals.

---

## Phase 2: Problem Framing

### The Core Tension

> "Engagement optimization is a **multi-timescale problem**. What maximizes *this session's* watch time may damage *this month's* retention. Clickbait gets clicks today but erodes trust over time. We need to decompose engagement into short-term and long-term components and optimize a blend."

### Redefining the Objective:

Instead of:
$$\max \sum_{t} \text{WatchTime}(\text{user}_i, \text{video}_t)$$

We want:
$$\max \sum_{t} \underbrace{\text{Satisfaction}(\text{user}_i, \text{video}_t)}_{\text{per-video quality}} + \lambda \cdot \underbrace{\text{RetentionProbability}(\text{user}_i, t+7d)}_{\text{long-term health}}$$

Where $$\text{Satisfaction}$$ is a learned proxy (not just watch time).

---

## Phase 3: System Architecture

### The Recommendation Funnel

```
~800M videos
     │
     ▼ [Candidate Generation: ~1000 candidates]
   Recall-optimized, cheap models
     │
     ▼ [Ranking: ~100 ranked items]
   Precision-optimized, expensive models
     │
     ▼ [Re-ranking / Policy Layer: final ~20 shown]
   Diversity injection, safety filters, business rules
```

### Sub-System A: Candidate Generation

**Goal**: Reduce 800M → ~1000 candidates. Must be fast (< 20ms).

**Approaches**:
- **Collaborative filtering**: Users who watched X also watched Y
- **Content-based**: Videos similar in topic/embedding to user's history
- **Graph-based**: Follow subscription chains, co-watch graphs
- **Trending/fresh**: Inject recently uploaded, fast-growing content

**Architecture**: Two-tower model (user tower + video tower). Pre-compute video embeddings. At serve time, compute user embedding and do approximate nearest neighbor (ANN) search.

$$\text{score}(u, v) = \langle \mathbf{u}, \mathbf{v} \rangle$$

Where $$\mathbf{u} = f_\theta(\text{user\_history})$$ and $$\mathbf{v} = g_\phi(\text{video\_features})$$.

### Sub-System B: Ranking Model

**Goal**: Score ~1000 candidates with a rich model. Can spend ~100ms total.

**Multi-Task Architecture** (predict multiple signals simultaneously):

$$\hat{y}_{\text{click}}, \hat{y}_{\text{watch\_frac}}, \hat{y}_{\text{like}}, \hat{y}_{\text{share}}, \hat{y}_{\text{satisfied}} = \text{MultiTaskNN}(\mathbf{x}_{u,v})$$

Where $$\mathbf{x}_{u,v}$$ includes user features, video features, and cross features.

**Combining predictions into a single score**:

$$\text{RankScore} = w_1 \cdot \hat{y}_{\text{click}} \cdot \hat{y}_{\text{watch\_frac}} + w_2 \cdot \hat{y}_{\text{like}} + w_3 \cdot \hat{y}_{\text{share}} + w_4 \cdot \hat{y}_{\text{satisfied}} - w_5 \cdot \hat{y}_{\text{regret}}$$

The weights $$w_1, ..., w_5$$ are tuned via A/B tests on long-term retention.

**Key insight**: Including a negative term for predicted *regret* (user clicks "not interested" or closes quickly) directly penalizes clickbait.

### Sub-System C: Re-Ranking / Policy Layer

**Goal**: Inject diversity, enforce policies, balance multiple objectives.

**Diversity injection**:
- Ensure no more than 2 videos from the same channel in top 10
- Ensure at least 3 distinct topic categories in top 10
- Implement Maximal Marginal Relevance (MMR):

$$\text{MMR}(v) = \lambda \cdot \text{RankScore}(v) - (1-\lambda) \cdot \max_{v' \in S} \text{sim}(v, v')$$

Where $$S$$ is the set of already-selected videos.

**Safety filters** (hard constraints):
- Demote videos from channels with recent policy strikes
- Remove age-restricted content for underage users
- Apply reduced recommendation for borderline content

---

## Phase 4: The Satisfaction Model (Deep-Dive)

### Why Watch Time Alone Fails:

| Scenario | Watch Time | True Satisfaction |
| --- | --- | --- |
| User watches 10-min clickbait, feels tricked | High | Low |
| User watches 2-min tutorial, solves their problem | Low | High |
| User binge-watches, later regrets time spent | Very High | Negative |
| User discovers a new creator they love | Medium | Very High |

### Building a Satisfaction Proxy:

**Training data**: Use post-video surveys ("Was this worth your time?" on a random sample of videos) as ground truth.

**Proxy features** (measurable without surveys):
- Fraction of video watched (completion rate)
- Engagement *after* watching: like, share, subscribe to channel
- Negative signals: "not interested" click, leaving YouTube entirely, going back to re-watch the video (confusion)
- Time to next session (short = good? Or short = came back compulsively?)

**The Satisfaction Label**:

$$\text{satisfaction}_i = \alpha_1 \cdot \text{liked}_i + \alpha_2 \cdot \text{shared}_i + \alpha_3 \cdot \text{subscribed}_i - \alpha_4 \cdot \text{not\_interested}_i + \alpha_5 \cdot \text{survey\_response}_i$$

Calibrate $$\alpha$$ weights so that the composite correlates maximally with survey responses.

---

## Phase 5: Addressing Filter Bubbles

### The Exploration-Exploitation Tradeoff in Recommendations:

> "If we only show users what they've liked before, we never learn what *else* they might like. And we narrow their worldview over time."

**Approach 1: Epsilon-Greedy Exploration**
- With probability $$\epsilon$$, replace a ranked item with a random or diverse item
- Simple but wastes exploration budget (random items are often irrelevant)

**Approach 2: Upper Confidence Bound (UCB)**

$$\text{score}(v) = \hat{\mu}_v + \beta \cdot \hat{\sigma}_v$$

Videos with high uncertainty ($$\hat{\sigma}_v$$) get a boost. New videos and underexplored topics automatically surface.

**Approach 3: Counterfactual Diversity**
- Track each user's **topic exposure distribution** over time
- If distribution becomes too concentrated (low entropy), actively inject items from underrepresented topics
- $$\text{DiversityBoost}(v) \propto \frac{1}{\text{UserExposure}(\text{topic}(v))}$$

---

## Phase 6: Training, Serving & Monitoring

### Training Challenges:

- **Position bias**: Items shown in position 1 get more clicks regardless of quality. Must correct for position during training:
$$P(\text{click}) = P(\text{examine} | \text{position}) \times P(\text{click} | \text{examine, relevance})$$

- **Selection bias**: We only observe outcomes for videos we *showed*. Videos never recommended have no engagement signal. Use inverse propensity weighting.

- **Feedback loops**: If the model promotes video X, X gets more views, more engagement data, model promotes X more. Need to break the loop with exploration and temporal holdouts.

### Serving Infrastructure:
- User embeddings: updated hourly in batch, cached in feature store
- Video embeddings: updated on upload + daily refresh
- ANN index: rebuilt every 6 hours (HNSW or ScaNN)
- Ranking model: served via GPU inference, batched scoring

### Monitoring:
- **Offline metrics (weekly)**: NDCG, recall@K, satisfaction AUC
- **Online metrics (daily)**: Watch time per session, sessions per week, satisfaction survey scores
- **Long-term metrics (monthly)**: 28-day retention, subscriber growth, topic diversity of consumption
- **Safety metrics (real-time)**: Fraction of impressions that are borderline content, user reports per 1M views

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes a single recommendation model | Designs the full funnel (candidate gen → ranking → re-ranking) with different optimization criteria at each stage |
| Optimizes for watch time | Proposes a multi-objective function that includes satisfaction, diversity, and penalizes regret |
| Mentions filter bubbles | Proposes concrete mechanisms (UCB exploration, diversity constraints, entropy-based monitoring) |
| Ignores serving constraints | Explicitly designs around 200ms latency budget with pre-computed embeddings and ANN search |
| Treats training as straightforward | Addresses position bias, selection bias, and feedback loops as first-class challenges |

# Case Study 8: LinkedIn — "Design People You May Know (PYMK) at Scale"

---

## The Problem Statement (As Given by the Interviewer)

> *"LinkedIn's 'People You May Know' feature drives the majority of network growth. Design an ML system that suggests relevant connections to users. The system must handle 900M+ users, serve recommendations in real-time, and balance relevance with network health (we don't want to just suggest celebrities to everyone)."*

---

## Phase 1: Clarifying Questions

### Questions That Signal Graph ML Understanding:

1. **"What does 'relevant' mean? People the user knows in real life? People who would be professionally useful? People likely to accept the connection?"**
   - Optimizing for acceptance rate vs. professional value vs. mutual benefit are very different objectives.

2. **"What's the primary success metric? Connection request sent? Connection accepted? Subsequent engagement after connecting?"**
   - Accepted connections that lead to zero interaction are hollow. The true metric might be "connections that result in at least one message exchange within 30 days."

3. **"What signals indicate someone 'knows' another person vs. just being in the same professional space?"**
   - Shared company history (at the same time), shared school + graduation year, email/phone contacts uploaded, mutual connections

4. **"What are the anti-goals? What should we explicitly NOT recommend?"**
   - Don't suggest the user's ex-boss they had a conflict with
   - Don't suggest competitors' employees to C-suite (information leakage concern)
   - Don't create a network where everyone connects to celebrities (degrades network signal)

5. **"What's the serving budget? How many candidates do we score per user per day?"**
   - With 900M users, pairwise scoring is $$O(n^2) \approx 10^{17}$$ — completely infeasible. Need extreme candidate filtering.

---

## Phase 2: Problem Framing

### The Core Problem

> "This is a **link prediction problem on a massive heterogeneous graph** with the constraint that we can only score a tiny fraction of possible pairs. The system is 90% about **efficient candidate generation** (finding the right needles in the haystack) and 10% about ranking those candidates."

### The Graph Structure:

- **Nodes**: Users (900M), Companies (50M), Schools (200K), Skills (50K)
- **Edges**: Connections, employment history, education, endorsements, group memberships, content interactions
- **Temporal**: Relationships have start/end times (same company overlap matters)

### Key Insight:

> "The signal for 'you know this person' is fundamentally different from 'you should know this person.' The former is about shared history (overlapping employment, same school year). The latter is about network utility (bridge to a new cluster, shared professional interest). We need to handle both."

---

## Phase 3: System Architecture

### The Multi-Source Candidate Generation Strategy

Since we can't score all 900M users, we need multiple **candidate sources** that each contribute a pool of likely matches:

| Source | Logic | Expected Quality | Volume |
| --- | --- | --- | --- |
| Friends of friends (FoF) | 2-hop graph traversal | Very high (triadic closure) | ~5,000 per user |
| Same company + time overlap | Employment graph join | High | ~500 per user |
| Same school + graduation year | Education graph join | High | ~200 per user |
| Imported contacts match | Email/phone → profile match | Very high | Variable |
| Similar profile (embedding) | ANN in embedding space | Medium | ~1,000 per user |
| Content co-engagement | Liked/commented on same posts | Medium-low | ~500 per user |

**Total candidates per user**: ~7,000-10,000 (after deduplication)

### Candidate Generation: Friends of Friends (FoF)

**Why FoF is so powerful**: Triadic closure — if A knows B and B knows C, there's a high probability A knows C.

$$P(\text{edge}(A,C)) \propto |\text{Mutual}(A,C)|^\alpha$$

Where $$|\text{Mutual}(A,C)|$$ = number of mutual connections. Empirically, the probability of knowing someone increases superlinearly with mutual connections ($$\alpha \approx 1.3$$).

**Implementation at scale**: 
- Pre-compute 2-hop neighborhoods in a daily batch job (MapReduce/Spark on the graph)
- For a user with 500 connections, each of whom has 500 connections: 2-hop set is up to 250K (with dedup, typically ~50K). Filter to top 5K by mutual count.

---

## Phase 4: The Ranking Model

### Feature Engineering

**Structural features** (from graph):
- Number of mutual connections
- Jaccard similarity of connection sets: $$J(A,C) = \frac{|N(A) \cap N(C)|}{|N(A) \cup N(C)|}$$
- Adamic-Adar index (weighted mutuals): $$\text{AA}(A,C) = \sum_{z \in N(A) \cap N(C)} \frac{1}{\log |N(z)|}$$
  (Mutuals who have few connections themselves are stronger signals)
- Graph distance (2 vs. 3+ hops)

**Professional overlap features**:
- Same current company (binary)
- Same past company + time overlap (continuous: months of overlap)
- Same school + $$|\text{grad year}_A - \text{grad year}_C| \leq 2$$
- Shared skills count
- Same industry

**Behavioral features**:
- Has user A viewed user C's profile? (strong intent signal)
- Has user A searched for C's name or company?
- Content interaction overlap (liked same posts)

**Temporal features**:
- How recently did mutual connections form? (Fresh mutuals suggest an evolving social circle)
- When did A and C last update their profiles? (Active users more likely to accept)

### Model Architecture:

**Gradient-boosted trees** (LightGBM) for the production ranker:
- Fast inference (must score ~7000 candidates per user within seconds)
- Handles mixed feature types well
- Provides feature importance for debugging

**Training data**: 
- Positive examples: Connection requests that were **accepted** (not just sent)
- Negative examples: Impressions (shown as PYMK) that received no action within 7 days
- Hard negatives: Sent requests that were **ignored** or **declined** (these are cases where the model thought it was a good match but the user disagreed)

### The Two-Stage Ranking:

**Stage 1 (Lightweight, scores all ~7000 candidates)**:
- Use only pre-computed features (mutual count, structural similarity, company overlap)
- Score threshold or top-500 filter

**Stage 2 (Heavy, scores ~500 candidates)**:
- Include behavioral features (profile views, search, interaction)
- Include real-time features (current session context)
- Produce final ranked list of ~50 suggestions

---

## Phase 5: Network Health & Anti-Patterns

### The Celebrity Problem:

> "If we optimize purely for acceptance probability, we'd suggest Barack Obama to everyone (he might accept some). But this degrades the network — celebrity connections carry no signal."

**Solution**: Penalize recommendations where the degree ratio is extreme:

$$\text{penalty}(A, C) = \max\left(0, \; \log\frac{\text{degree}(C)}{\text{degree}(A)} - \tau\right)$$

If C has 10,000x more connections than A, the recommendation is likely not mutual benefit.

### The Echo Chamber Problem:

> "If we only suggest people similar to the user's existing network, we reinforce homogeneity."

**Solution**: Inject a small percentage of "bridge" recommendations — people in adjacent professional communities who share *one* strong connection but are otherwise in different clusters.

Identify bridges using network community detection (Louvain). Recommend cross-community connections when there's at least one strong mutual.

### The Stale Network Problem:

> "Users who haven't been active in months still appear in PYMK for others. They'll never accept."

**Solution**: Weight candidates by recency of activity:
$$\text{ActivityWeight}(C) = e^{-\lambda \cdot \text{days\_since\_last\_active}(C)}$$

---

## Phase 6: Evaluation & Deployment

### Offline Metrics:
- **Precision@K**: Of top K suggestions, what fraction were accepted?
- **Recall**: Of all connections formed this month, what fraction were surfaced by PYMK?
- **Reciprocal Rank**: How high in the list was the first accepted suggestion?

### Online Metrics (A/B test):
- **Primary**: Accepted connections per user per week
- **Secondary**: Messages exchanged with new connections (measures connection *quality*)
- **Guardrail**: Decline rate (if recommendations are bad, users explicitly decline more)
- **Long-term**: Network density, session frequency, content engagement

### Cold Start:
- New users with no connections: use profile features (school, company, location) to match with high-confidence FoF candidates from those institutions
- After first 5 connections: FoF becomes the dominant source
- Prompt user to import contacts (phone, email) — this is the strongest cold-start signal

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes collaborative filtering on the whole graph | Recognizes the computational infeasibility and designs a multi-source candidate generation strategy |
| Uses mutual connections as a feature | Uses Adamic-Adar and explains WHY weakly-connected mutuals are stronger signals |
| Trains on sent requests | Trains on *accepted* requests and uses ignored requests as hard negatives |
| Optimizes for acceptance rate | Discusses network health (celebrity problem, echo chambers) and includes penalties |
| Ignores cold start | Designs explicit cold-start pathways and progressive feature availability |

# Case Study 9: Amazon — "Design a Demand Forecasting System for Inventory Optimization"

---

## The Problem Statement (As Given by the Interviewer)

> *"Amazon manages 600M+ SKUs across hundreds of fulfillment centers worldwide. We need to forecast demand for each SKU at each location over multiple time horizons (daily for 14 days, weekly for 12 weeks) to optimize inventory placement. Under-stocking means lost revenue and unhappy customers. Over-stocking means warehousing costs and product expiration. Design this system."*

---

## Phase 1: Clarifying Questions

### Questions That Reveal Supply Chain + ML Understanding:

1. **"What's the loss asymmetry? For each SKU category, what's the cost ratio of a stockout vs. overstock?"**
   - Perishable goods (food): overstock → waste (high cost). Stockout → customer goes elsewhere.
   - Electronics: overstock → carrying cost + depreciation. Stockout → high revenue loss (expensive items).
   - Long-tail items: overstock → indefinite storage. Stockout → minimal impact (few buyers).
   - This determines whether we optimize mean forecast or quantile forecast.

2. **"What's the distribution of demand patterns? How many SKUs are 'fast movers' vs. 'slow movers' (intermittent demand)?"**
   - ~80% of SKUs sell < 1 unit/day at any given FC. Standard time-series methods fail for intermittent demand.

3. **"What external signals do we have? Promotions, pricing changes, Prime Day events, weather, seasonality?"**
   - Demand is not just a function of history — it's driven by covariates we can observe ahead of time.

4. **"What's the lead time for replenishment? Is it the same for all products?"**
   - If lead time is 2 days, we need 2-day-ahead forecasts. If it's 6 weeks (international shipping), we need 6-week-ahead forecasts. Different horizons need different approaches.

5. **"How many distinct forecasts are we producing? (SKU × location × horizon)"**
   - 600M SKUs × 200 FCs × 14 daily forecasts = trillions of predictions. This is an extreme-scale ML serving challenge.

---

## Phase 2: Problem Framing

### The Core Insight

> "This is not one forecasting problem. It's at least THREE distinct problems:
> 1. **High-volume SKUs** (top 5% by revenue): standard time series, rich history, worth custom attention
> 2. **Mid-volume SKUs** (next 30%): enough data for ML but not enough for per-SKU models
> 3. **Long-tail / intermittent demand** (bottom 65%): sparse data, need hierarchical or zero-inflated models

> The system architecture must handle all three, not force one approach on all."

### Why Traditional Time Series Fails Here:

- ARIMA/ETS work for single series with sufficient history. At 600M SKUs, we can't fit 600M individual models.
- New SKUs (cold start) have zero history.
- Demand is driven by promotions, pricing, and events that aren't in the history.
- Intermittent demand violates the continuity assumptions of most time-series methods.

---

## Phase 3: System Architecture

### The Hierarchical Forecasting Approach

```
Level 1: Category-level (Electronics, Grocery, Clothing)
     │
 Level 2: Sub-category (Laptops, Headphones, Cables)
     │
 Level 3: Brand-level (Apple Laptops, Dell Laptops)
     │
 Level 4: SKU-level (MacBook Air M3 256GB Space Gray)
     │
 Level 5: SKU × Location (MacBook Air M3 @ Seattle FC)
```

**Key principle**: Higher levels have more data (more stable forecasts). Lower levels have more specificity. The system should **reconcile** forecasts across levels.

### Forecast Reconciliation:

$$\hat{y}_{\text{reconciled}} = S \cdot P \cdot \hat{y}_{\text{base}}$$

Where:
- $$S$$ is the summing matrix (maps bottom-level to all aggregation levels)
- $$P$$ is a projection matrix (optimally combines forecasts from all levels)
- Common methods: MinT (Minimum Trace), bottom-up, top-down proportions

---

## Phase 4: Model Design

### The Global Model Approach (For Mid/Long-Tail SKUs)

**Key idea**: Instead of fitting millions of individual models, train ONE model on ALL SKUs simultaneously, using SKU features as inputs.

$$\hat{y}_{s,t+h} = f(\text{history}_{s,t}, \text{SKU\_features}_s, \text{covariates}_{s,t+h})$$

**Model architecture** (DeepAR / Temporal Fusion Transformer style):

**Input features**:
- **Time-varying known future**: day of week, month, holiday indicators, planned promotions, price
- **Time-varying observed past**: historical demand, returns, page views, add-to-cart rate
- **Static (SKU-level)**: category, brand, price tier, weight, dimensions, launch date
- **Location-level**: FC capacity, region population, local events

**Why global models work here**:
- A new headphone SKU can borrow patterns from similar headphones that launched before
- Seasonal patterns are shared across SKUs in the same category
- Promotion effects are transferable ("20% off" effect generalizes)

### For High-Volume SKUs: Specialized Models

Top 5% of SKUs by revenue get dedicated attention:
- Individual LightGBM models with extensive feature engineering
- Ensemble with the global model (average the two forecasts)
- Human forecaster override for major events (Prime Day, Black Friday)

### For Intermittent Demand: Zero-Inflated Models

SKUs that sell < 1/day require a two-stage model:

$$P(\text{demand}_t > 0) = \text{logistic}(\mathbf{x}_t)$$
$$E[\text{demand}_t | \text{demand}_t > 0] = \exp(\mathbf{x}_t^\top \beta)$$

Combined forecast:
$$E[\text{demand}_t] = P(\text{demand}_t > 0) \times E[\text{demand}_t | \text{demand}_t > 0]$$

Alternative: Croston's method or its variants (SBA, TSB) for intermittent series.

---

## Phase 5: From Point Forecasts to Inventory Decisions

### The Critical Insight Most Candidates Miss:

> "The business doesn't need a *point forecast*. It needs a *decision*: how many units to stock. This requires **probabilistic forecasts** (prediction intervals), not just the mean."

### The Newsvendor Problem:

Optimal stock level $$q^*$$ satisfies:

$$F(q^*) = \frac{c_u}{c_u + c_o}$$

Where:
- $$c_u$$ = cost of one unit of under-stock (lost sale + customer dissatisfaction)
- $$c_o$$ = cost of one unit of over-stock (carrying cost + potential waste)
- $$F$$ = CDF of the demand forecast distribution

**Example**: If under-stock costs 5x more than over-stock:
$$F(q^*) = \frac{5}{5+1} = 0.833$$

We should stock at the 83rd percentile of the forecast distribution, not the mean.

### Producing Probabilistic Forecasts:

**Option 1**: Quantile regression — directly predict the 10th, 50th, 90th percentiles

$$\mathcal{L}_{\text{quantile}}(y, \hat{y}_q) = \begin{cases} q \cdot (y - \hat{y}_q) & \text{if } y \geq \hat{y}_q \\ (1-q) \cdot (\hat{y}_q - y) & \text{if } y < \hat{y}_q \end{cases}$$

**Option 2**: Distributional forecast — predict parameters of a distribution (e.g., Negative Binomial for count data)

$$\text{demand} \sim \text{NegBin}(\mu_{s,t}, \alpha_{s,t})$$

Where both $$\mu$$ and $$\alpha$$ are outputs of the neural network.

---

## Phase 6: Evaluation & Monitoring

### Offline Evaluation Metrics:

| Metric | What It Measures | When to Use |
| --- | --- | --- |
| MAPE (Mean Absolute Percentage Error) | Relative forecast error | High-volume SKUs |
| WAPE (Weighted Absolute Percentage Error) | Error weighted by actual demand | Avoids divide-by-zero for low-volume |
| Quantile Loss (Pinball Loss) | Calibration of probabilistic forecasts | Always (for inventory decisions) |
| Coverage (% of actuals within prediction interval) | Interval calibration | Should be ~90% for 90% interval |

### The Backtest Protocol:

- Sliding window: train on months 1-6, forecast month 7. Train on 2-7, forecast 8. Etc.
- **Never** include future promotions/events in training that wouldn't have been known at forecast time
- Evaluate separately by demand pattern (fast/mid/intermittent) — aggregate metrics hide problems

### Business Metrics (What Actually Matters):

- **Fill rate**: % of customer orders fulfilled from in-stock inventory
- **Inventory turns**: revenue / average inventory value (efficiency)
- **Dead stock rate**: % of inventory with zero sales in 90 days
- **Expedited shipping cost**: when we stockout and must ship from a distant FC

### Monitoring in Production:

- **Forecast bias tracking**: Are we systematically over- or under-forecasting? By category? By FC?
$$\text{Bias} = \frac{\sum (\hat{y} - y)}{\sum y}$$
- **Drift detection**: Monitor feature distributions (especially promotion features and price) for unexpected changes
- **Alert on stockouts**: When actual demand exceeds the 95th percentile forecast for 3+ consecutive days

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes one time-series model | Recognizes 3+ distinct demand patterns requiring different approaches |
| Trains individual models per SKU | Proposes a global model that shares information across SKUs |
| Produces point forecasts | Explains why probabilistic forecasts are essential for inventory decisions and derives the newsvendor quantile |
| Evaluates with MAPE | Uses quantile loss, evaluates separately by demand pattern, and connects to business metrics (fill rate, inventory turns) |
| Ignores cold start | Designs the global model specifically to handle new SKUs via feature transfer |

# Case Study 10: X (Twitter) — "Design a Content Safety System That Doesn't Kill Engagement"

---

## The Problem Statement (As Given by the Interviewer)

> *"X's timeline ranking currently optimizes for engagement (likes, retweets, replies, dwell time). But engagement-optimized feeds amplify toxic, divisive, and misleading content because such content generates strong reactions. Design a system that demotes harmful content in the feed WITHOUT significantly reducing overall engagement. The CEO insists there should be zero impact on 'healthy' engagement."*

---

## Phase 1: Clarifying Questions

### Questions That Reveal Content Moderation + ML Nuance:

1. **"How do we define 'harmful'? Is there a taxonomy? Is it binary (harmful/not) or graded (severity levels)?"**
   - Taxonomy might include: hate speech, misinformation, harassment, violence, spam, self-harm, graphic content
   - Each category has different signals, severity levels, and cultural variation

2. **"Are we talking about removing/blocking content OR reducing its distribution (demotion)? These are very different systems."**
   - Removal = binary enforcement decision (high bar, appeals process, legal implications)
   - Demotion = ranking decision (softer, probabilistic, reversible)
   - This interview is about *demotion in ranking*, not removal.

3. **"What's 'healthy' engagement? Can we operationalize the CEO's requirement?"**
   - This is the crux. If toxic content gets 3x replies but those replies are angry arguments, is that "engagement" we want?
   - Need to define healthy vs. unhealthy engagement *before* building the system.

4. **"What labeled data do we have? Human-annotated content? User reports? Automated classifiers?"**
   - Human annotation is expensive and slow. User reports are biased (weaponized reporting). Existing classifiers have known failure modes.

5. **"Are there legal/regulatory constraints? Section 230 implications? International content moderation laws vary dramatically."**
   - EU DSA requires certain content actions. US has different norms. Must be jurisdiction-aware.

---

## Phase 2: Problem Framing

### The Multi-Objective Ranking Problem

> "This is NOT a classification problem (is this content harmful?). It's a **multi-objective ranking problem**: given a set of candidate tweets for a user's timeline, rank them to maximize a blended objective that rewards genuine engagement while penalizing toxicity-driven engagement."

### Decomposing "Engagement" Into Healthy vs. Unhealthy:

| Signal | Likely Healthy | Likely Unhealthy |
| --- | --- | --- |
| Like + share | Appreciation, endorsement | Engagement with outrage content |
| Reply | Conversation, discussion | Angry pile-on, harassment |
| Quote tweet | Commentary, amplification | Mockery, dunking |
| Dwell time | Reading, absorbing | Rubbernecking at a train wreck |
| Follow author after reading | Discovery of quality voice | Following divisive figure reactively |

**Key insight**: The same *action* (reply) can be healthy or unhealthy depending on *content and context*. We need to model the *quality* of engagement, not just its presence.

### The Objective Function:

$$\text{Score}(\text{tweet}) = \underbrace{P(\text{engage}) \cdot V(\text{engage})}_{\text{Expected engagement value}} - \underbrace{\alpha \cdot P(\text{harmful}) \cdot S(\text{severity})}_{\text{Harm penalty}}$$

Where:
- $$V(\text{engage})$$ = value of the engagement (likes/shares weighted higher than angry replies)
- $$P(\text{harmful})$$ = probability the content is harmful
- $$S(\text{severity})$$ = severity if harmful (hate speech > mild incivility)
- $$\alpha$$ = business parameter controlling the safety-engagement tradeoff

---

## Phase 3: System Architecture

### The Three-Layer Safety Integration:

```
Layer 1: Content Understanding (Offline/Near-real-time)
   Tweet text, images, video → safety scores, topic classification
       │
 Layer 2: Engagement Prediction (Real-time)
   User × Tweet features → P(like), P(reply), P(share), P(dwell > 30s)
       │
 Layer 3: Blended Ranking (Real-time)
   Combine engagement predictions with safety scores + diversity constraints
       │
   Final timeline
```

### Layer 1: Content Safety Scoring

**Multi-label classifier** (not binary!):

For each tweet, produce:
$$[P(\text{hate}), P(\text{misinfo}), P(\text{harassment}), P(\text{violence}), P(\text{spam}), P(\text{self-harm})]$$

Plus a severity estimate for each:
$$[S(\text{hate}), S(\text{misinfo}), ...]$$

**Model**: Fine-tuned language model (BERT-base or smaller for latency) on human-annotated data.

**Challenges**:
- Sarcasm, irony, coded language ("Let's go Brandon" — political coding)
- Context-dependent harm (medical discussion vs. promotion of self-harm)
- Multimodal content (text says one thing, image says another)
- Multilingual content (English model fails on Hindi/Arabic/etc.)

### Layer 2: Engagement Quality Prediction

**Beyond binary engagement, predict QUALITY of engagement**:

$$P(\text{positive\_reply}) \text{ vs. } P(\text{negative\_reply})$$

Train a reply quality classifier:
- Positive: agreement, thanks, additional info, humor
- Negative: insults, threats, sarcasm, personal attacks

Use this to compute expected engagement *value*:

$$V(\text{engage}) = w_1 \cdot P(\text{like}) + w_2 \cdot P(\text{share}) + w_3 \cdot P(\text{positive\_reply}) - w_4 \cdot P(\text{negative\_reply})$$

### Layer 3: The Final Ranking

**Constrained optimization**: Rank by blended score subject to:
- No more than 1 tweet with $$P(\text{harmful}) > 0.3$$ in top 10
- Timeline-level diversity (topic, author, political leaning)
- At most 20% engagement reduction vs. engagement-only ranking (the CEO's constraint)

---

## Phase 4: The Calibration Challenge (Deep-Dive)

### Why This Is Harder Than It Sounds:

> "The problem with demoting harmful content is that **we don't know the counterfactual**. If we demote a viral toxic tweet, engagement on the platform drops. But is that because we removed 'engagement' (bad engagement we don't want) or because users who came for outrage are leaving (which might be fine)?"  

### The Measurement Framework:

Decompose total engagement change:

$$\Delta E_{\text{total}} = \underbrace{\Delta E_{\text{on\_demoted}}}_{\text{Expected: negative}} + \underbrace{\Delta E_{\text{on\_other}}}_{\text{Key question}}$$

- If $$\Delta E_{\text{on\_other}} > 0$$: demoting toxic content *increases* engagement on healthy content (users engage more when feed is cleaner). **Net win.**
- If $$\Delta E_{\text{on\_other}} \approx 0$$: engagement just shifts away from toxic content, no harm to healthy. **Acceptable.**
- If $$\Delta E_{\text{on\_other}} < 0$$: some users disengage entirely when denied toxic content. **The hard case.** 

### Experimental Strategy:

**Gradual rollout with ramps**:
- $$\alpha = 0.0$$ (baseline: no safety demotion)
- $$\alpha = 0.1, 0.3, 0.5, 1.0$$ (increasing safety weight)

For each $$\alpha$$ level, measure:
- Total DAU, sessions per DAU, time per session
- Engagement on non-demoted content (the healthy baseline)
- User satisfaction surveys ("Is your feed improving?")
- Report rate ("see fewer reports" = users encountering less harm)

Find the $$\alpha^*$$ that maximizes:
$$\alpha^* = \arg\max_\alpha \; [\text{Healthy\_Engagement}(\alpha) - \text{Safety\_Cost}(\alpha)]$$

Subject to: total engagement drop < 5%.

---

## Phase 5: The Feedback Loop & Adversarial Dynamics

### How Bad Actors Adapt:

- **Coded language**: Replace slurs with coded terms (new vocabulary)
- **Engagement farming**: Craft content that's technically "borderline" but triggers strong reactions
- **Coordinated behavior**: Multiple accounts amplify each other to boost rankings
- **Adversarial examples**: Add innocuous text/images to confuse the safety classifier

### Defense Strategies:

1. **Continuous retraining** on newly discovered harm patterns (weekly model updates)
2. **Behavioral signals over content signals**: accounts that consistently generate angry replies should be downranked regardless of what their text says
3. **Network-level features**: tweet from account that shares followers with many suspended accounts → higher prior for harm
4. **Human-in-the-loop**: borderline cases go to content moderators; their decisions become training data

### The Author-Level Reputation Score:

$$\text{AuthorTrust}_a = \sigma\left(\beta_0 + \beta_1 \cdot \text{account\_age} + \beta_2 \cdot \text{violation\_history} + \beta_3 \cdot \text{avg\_reply\_quality} + \beta_4 \cdot \text{follower\_legitimacy}\right)$$

This acts as a prior that adjusts the content-level safety score:

$$P(\text{harmful} | \text{tweet, author}) = P(\text{harmful} | \text{tweet}) \cdot (1 + \gamma \cdot (1 - \text{AuthorTrust}))$$

---

## Phase 6: Evaluation & The Philosophical Challenge

### The Fundamental Tension:

> "There is no objective ground truth for 'harmful.' What's harmful depends on context, culture, political perspective, and evolving social norms. Any system we build embeds value judgments. The question is: whose values, and how are they updated?"

### How to Handle This in an Interview:

1. **Acknowledge the subjectivity** explicitly. Don't pretend it's a purely technical problem.
2. **Propose a governance structure**: who sets the taxonomy? How are edge cases adjudicated? How often is the policy reviewed?
3. **Build appeals and transparency**: users whose content is demoted should be able to understand why and appeal.
4. **Measure fairness**: does the system disproportionately demote content from certain demographics, political orientations, or languages?

### Fairness Metrics:

- False positive rate by author demographic / political leaning
- Demotion rate by language (non-English often has worse classifier performance)
- Appeal success rate by group (if one group's appeals succeed 50% of the time, the model is biased against them)

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Builds a toxicity classifier | Designs a multi-objective ranking system where toxicity is one input among many |
| Treats engagement as uniform | Decomposes engagement into healthy vs. unhealthy and models quality of interaction |
| Proposes binary harmful/not | Produces graded severity scores and acknowledges context-dependence |
| Ignores measurement | Proposes experimental framework to calibrate the safety-engagement tradeoff with concrete metrics |
| Treats it as purely technical | Acknowledges the value-laden nature of content moderation and proposes governance structures |

# Case Study 11: Waymo — "Design the Perception System for Detecting Vulnerable Road Users"

---

## The Problem Statement (As Given by the Interviewer)

> *"Waymo's autonomous vehicles must detect pedestrians, cyclists, and other vulnerable road users (VRUs) in all conditions. A miss can be fatal. Design the ML perception system for VRU detection. The system must operate at 10 Hz (100ms per frame) with > 99.9% recall on VRUs within 50 meters while maintaining acceptable false positive rates in complex urban environments."*

---

## Phase 1: Clarifying Questions

### Questions That Reveal Safety-Critical ML Thinking:

1. **"What sensor suite is available? Camera-only, or multi-modal (lidar + radar + camera)?"**
   - Camera: rich semantic information, struggles in darkness/glare
   - Lidar: precise 3D geometry, struggles with range/weather
   - Radar: velocity information, works in all weather, low resolution
   - The answer shapes whether this is 2D detection, 3D detection, or fusion.

2. **"What's the operational design domain (ODD)? Urban only? Highways? All weather? Night?"**
   - Urban pedestrian detection at night in rain is 10x harder than daytime highway.
   - Each condition may need specialized models or thresholds.

3. **"What is 99.9% recall measured against? Per-frame or per-track? What counts as a 'detection'?"**
   - Per-frame: must detect in every single frame (very strict)
   - Per-track: must detect the object in at least K out of N frames (more realistic)
   - Detection criteria: IoU > 0.5? Center within 2m? Any partial detection?

4. **"What's the downstream consumer of detections? Does it need classification (pedestrian vs. cyclist) or just 'something is there'?"**
   - Planning needs: position, velocity, heading, classification, and **intent prediction** (will they cross?)

5. **"What's the acceptable false positive rate? Every FP can cause an unnecessary hard brake or lane change."**
   - In complex urban scenes (parking lots, street furniture), FPs are more common. A system that brakes for every trash can is undriveable.

---

## Phase 2: Problem Framing

### The Safety-Critical Framing

> "This is not an academic object detection problem. It's a **safety-critical decision support system** where:
> 1. False negatives are potentially fatal (hit a pedestrian)
> 2. False positives degrade ride quality and trust (phantom braking)
> 3. The system operates in an open world (encounters objects never seen in training)
> 4. Failure modes must be characterized and bounded (not just average performance)"

### The Key Architectural Principle:

> "No single model should be trusted with life-safety decisions. We need **redundancy** — multiple independent detection pathways that cross-check each other. The system should be designed so that **any single model failure does not result in a miss**."

---

## Phase 3: System Architecture

### Multi-Sensor Fusion Strategy

| Sensor | Strength | Weakness | Role |
| --- | --- | --- | --- |
| Camera (6-8 cameras, 360°) | Semantics, classification, lane lines | Depth estimation, night/glare | Primary classification |
| Lidar (1-3 units, 360°) | Precise 3D geometry, distance | Sparse at range, rain/fog | Primary 3D localization |
| Radar (4-6 units) | Velocity, all-weather | Low resolution, clutter | Velocity estimation, backup detection |

### Fusion Approaches:

**Early fusion** (raw sensor data combined before detection):
- Project lidar points onto camera image, create enriched input
- Pro: model learns cross-modal features. Con: tightly coupled, single point of failure.

**Late fusion** (each sensor produces independent detections, fuse at output):
- Camera detector + Lidar detector + Radar detector → Association + Fusion
- Pro: redundancy (if one sensor fails, others still detect). Con: may miss objects at sensor boundaries.

**Mid-level fusion** (share features between modalities at intermediate layers):
- Best accuracy but complex architecture.

**For safety-critical systems: late fusion with early fusion as primary.**
- Primary path: early fusion model (camera + lidar) for best accuracy
- Backup path: independent camera-only and lidar-only detectors
- If *any* path detects a VRU with high confidence, it's treated as a real detection

---

## Phase 4: Model Design (Deep-Dive)

### The Primary Detection Model (Lidar + Camera Fusion)

**Architecture**: BEV (Bird's Eye View) representation

1. Process camera images with a backbone (ResNet/EfficientNet) → multi-scale features
2. Lift camera features to 3D using depth estimation (LSS-style lift-splat-shoot)
3. Process lidar point cloud with a sparse 3D backbone (VoxelNet / PointPillars)
4. Combine in BEV space (top-down grid at 0.2m resolution)
5. Apply detection head: predict bounding boxes, classifications, velocities

**Output per detection**:
- 3D bounding box (x, y, z, length, width, height, heading)
- Classification: pedestrian / cyclist / motorcyclist / wheelchair / other VRU
- Velocity vector (vx, vy)
- Confidence score
- Uncertainty estimate (aleatoric + epistemic)

### Training Strategy:

**Data**: Millions of frames from fleet driving, manually annotated with 3D bounding boxes.

**Hard example mining**: Over-sample rare but critical scenarios:
- Partially occluded pedestrians
- Children (smaller, unexpected behavior)
- Pedestrians in dark clothing at night
- Cyclists emerging from behind parked cars
- Wheelchair users, people with strollers

**Loss function with asymmetric weighting**:

$$\mathcal{L} = \underbrace{\mathcal{L}_{\text{box}}}_{\text{localization}} + \underbrace{\lambda_{\text{cls}} \cdot \mathcal{L}_{\text{cls}}}_{\text{classification}} + \underbrace{\lambda_{\text{vel}} \cdot \mathcal{L}_{\text{velocity}}}_{\text{motion}}$$

With class-weighted focal loss for classification:

$$\mathcal{L}_{\text{cls}} = -\alpha_c (1 - p_t)^\gamma \log(p_t)$$

Where $$\alpha_{\text{VRU}} \gg \alpha_{\text{vehicle}}$$ (VRU misses are much more costly).

---

## Phase 5: Achieving 99.9% Recall

### Why 99.9% Is Extremely Hard:

- At 10 Hz for 1 hour of driving: 36,000 frames
- 99.9% recall per frame means ~36 missed frames per hour
- If a pedestrian is in view for 3 seconds (30 frames), 99.9% per-frame means ~97% chance of detecting in at least one frame
- For TRUE safety: need 99.99%+ per-track recall

### Strategies to Push Recall:

**1. Multi-scale detection**: Use feature pyramid networks to detect VRUs at all distances (small far-away pedestrians are hardest)

**2. Temporal integration**: A pedestrian missed in frame $$t$$ was likely detected in frame $$t-1$$. Use tracking to propagate detections:

$$P(\text{object exists at } t) = \max(P_{\text{detect}, t}, \; P_{\text{track}, t-1} \cdot (1 - \delta))$$

**3. Multiple operating points**: Run the detector at a very low threshold (high recall, many FPs) and use downstream tracking + velocity consistency to prune FPs.

**4. Radar as a safety net**: Radar reliably detects moving objects. Any radar return with pedestrian-like velocity (1-8 m/s) that doesn't match a known detection triggers a "CHECK" signal to other modalities.

**5. Uncertainty-aware planning**: When the detector is uncertain (low confidence but non-zero), the planning system should act conservatively (slow down) even without a confirmed detection.

---

## Phase 6: Validation & Safety Assurance

### The Evaluation Challenge:

> "You cannot prove 99.9% recall from road testing alone. To demonstrate 99.9% reliability with 95% confidence, you'd need ~3000 true positives without a miss. Given that VRU encounters per mile are low, this requires millions of miles."

### Multi-Pronged Validation Strategy:

**1. Simulation** (large scale, synthetic):
- Generate millions of synthetic scenarios in simulation
- Vary lighting, weather, occlusion, VRU appearance, behavior
- Inject adversarial scenarios (person lying on road, child running from behind car)

**2. Replay** (real data, offline):
- Replay real sensor data through the model
- Measure recall/precision on held-out annotated data
- Focus analysis on **failure modes**: when does it miss? Systematic patterns?

**3. Closed-course testing**:
- Physical test scenarios with mannequins and controlled conditions
- NHTSA-style test protocols (crossing pedestrians at various speeds/occlusions)

**4. Fleet monitoring** (deployed, continuous):
- Every time the planning system brakes for a VRU: log and review
- Every near-miss (VRU detected late): root-cause analysis
- Continuous recall estimation from random audit of fleet data

### Failure Mode Analysis:

Systematically catalog:
- At what distance do misses occur? (degradation curve vs. range)
- Which occlusion levels cause misses? (fully visible vs. 50% occluded vs. 75% occluded)
- Which weather/lighting conditions degrade performance? (quantify the degradation)
- Are there demographic biases? (darker skin tones in low light, children vs. adults)

---

## What Separates Good from Great Here:

| Good Candidate | Great Candidate |
| --- | --- |
| Proposes a single object detection model | Designs redundant multi-sensor pathways with independent failure modes |
| Reports average precision | Discusses per-frame vs. per-track recall, recognizes that 99.9% isn't achievable from testing alone, and proposes simulation + replay + fleet monitoring |
| Trains on standard datasets | Discusses hard example mining for rare safety-critical scenarios and asymmetric loss |
| Outputs bounding boxes | Outputs confidence + uncertainty estimates and explains how planning uses uncertainty |
| Treats this as a standard CV problem | Frames it as a safety-critical system with redundancy, graceful degradation, and validation methodology |

# Synthesis: Meta-Patterns Across All Ten Cases

---

## The 7 Thinking Patterns That Recur Across Every Case

### Pattern 1: Always Ask "Compared to What?"

Every case requires a **counterfactual**:
- Netflix: What would viewership be without promotion?
- Uber: What would rider experience be without surge?
- Meta: What does organic coordination look like?
- Airbnb: What would happen to a new host without the boost?
- Spotify: What would the metric be without the feature?
- Google Pay: What would fraud rates be without this model?
- YouTube: What would retention be without the recommendation?
- LinkedIn: What connections would form organically without PYMK?
- Amazon: What would demand be absent the promotion?
- X/Twitter: What would engagement be without demotion?

> *"The quality of your analysis is bounded by the quality of your counterfactual."*

---

### Pattern 2: Distinguish Causation from Correlation (Explicitly)

In every case, the naive analysis conflates correlation with causation:
- Netflix: Views dropped after launch ≠ launch caused the drop
- Uber: Surge areas have long waits ≠ surge causes long waits
- Meta: Accounts post similar content ≠ they're coordinated
- Airbnb: Unboosted listings don't perform ≠ boosting would help them
- Spotify: Treatment users listen more ≠ the feature made them listen more

Great candidates state the **identification strategy** explicitly: "Here's why I believe this relationship is causal, and here are the threats to that claim."

---

### Pattern 3: Multi-Metric, Multi-Stakeholder Thinking

No real problem has a single metric:

| Case | Stakeholder 1 | Stakeholder 2 | Hidden Stakeholder |
| --- | --- | --- | --- |
| Netflix | Viewers (engagement) | Content team (catalog utilization) | Licensed content partners (contractual minimums) |
| Uber | Riders (price, wait time) | Drivers (earnings, utilization) | Regulators (fairness) |
| Meta | Users (safety) | Advertisers (brand safety) | Civil society (democracy) |
| Airbnb | Guests (booking quality) | Hosts (visibility, earnings) | Communities (housing supply) |
| Spotify | Listeners (enjoyment) | Artists (exposure, royalties) | Advertisers (targeting quality) |
| Google Pay | Users (frictionless payments) | Risk team (loss prevention) | Merchants (false decline = lost sale) |
| YouTube | Viewers (satisfaction) | Creators (reach, revenue) | Society (information quality) |
| LinkedIn | Users (relevant connections) | Platform (network density) | Recruiters (signal quality of network) |
| Amazon | Customers (availability) | Operations (warehouse costs) | Sellers (fair inventory allocation) |
| X/Twitter | Users (safe experience) | Creators (reach) | Advertisers (brand safety) |

---

### Pattern 4: State Assumptions Explicitly, Then Stress-Test Them

Great candidates say:
- "This approach assumes X. Let me check if X holds."
- "If X doesn't hold, here's how my answer changes."

Examples:
- "Diff-in-diff assumes parallel trends. Let me verify with pre-period data."
- "The demand model assumes rational riders. In reality, riders may have reference-dependent preferences (surge at 2.0x feels unfair even if objectively reasonable)."
- "Thompson Sampling assumes independent draws. But listings compete for the same guests — showing one affects the other."

---

### Pattern 5: Think About the Adversary / Feedback Loop

| Case | The Feedback Loop |
| --- | --- |
| Netflix | Algorithm promotes what's popular → it becomes more popular → algorithm promotes more (rich-get-richer) |
| Uber | Drivers learn surge patterns → game the system → surge becomes less informative |
| Meta | Detection published → adversaries adapt → detection degrades |
| Airbnb | Top hosts get reviews → rank higher → get more bookings → more reviews (winner-take-all) |
| Spotify | Feature optimizes for time spent → safe recommendations → less discovery → long-term boredom |
| Google Pay | Blocked transactions never get labels → model never learns it was wrong → selection bias |
| YouTube | Model promotes engaging content → creators optimize for engagement signals → clickbait arms race |
| LinkedIn | PYMK suggestions form connections → creates new FoF candidates → network becomes what PYMK made it |
| Amazon | Stockouts of popular items → demand appears lower → model forecasts even less → perpetual stockout |
| X/Twitter | Demoting content reduces engagement data → model has less signal → harder to distinguish borderline from safe |

> *"If your solution doesn't account for how agents (users, adversaries, markets) will respond to it, you've solved a static problem in a dynamic world."*

---

### Pattern 6: Time Horizon Awareness

Always ask: **"Is this true in the short run, the medium run, and the long run?"**

- Short-term gains can mask long-term damage (novelty effects, marketplace concentration)
- Short-term costs can enable long-term gains (exploration, new host investment)
- The right answer at week 1 might be wrong at month 6

---

### Pattern 7: Propose, Don't Just Analyze

The interview isn't just diagnostic — it's prescriptive. End with:

1. **A clear recommendation** (ship / don't ship / ship with modifications)
2. **A confidence level** ("I'm 80% confident because...")
3. **A monitoring plan** ("Here's how we'd know if we were wrong")
4. **A reversibility plan** ("If this fails, here's the rollback strategy")

---

## Final Thought: The Interview Is Not About the Answer

> *The interviewer already knows the answer. They're watching HOW you get there. Do you:*
> - *Structure before diving in?*
> - *Acknowledge uncertainty rather than overfit to one solution?*
> - *Connect technical methods to business impact?*
> - *Communicate tradeoffs rather than pretending there's one right answer?*
> - *Know what you don't know, and say so?*
>
> *The candidate who says "I'd need to check X before committing to this approach" beats the candidate who confidently presents a wrong solution every time.*